# Tri-Stream DeepFake Detection — Google Colab
**Tự chứa hoàn toàn** — chỉ cần upload file `.ipynb` này lên Colab, chạy từ trên xuống.

**Luồng:**  
1. Cài dependencies  
2. Mount Drive → cấu hình đường dẫn  
3. Ghi code dự án ra đĩa (tự động)  
4. Tạo split manifest từ CSV  
5. Extract face crops → lưu vào Drive của bạn  
6. Train model  
7. Evaluate

In [1]:
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


In [2]:
import sys

# First, uninstall conflicting packages if they were partially installed by previous attempts or different versions.
!pip uninstall -y numpy pandas

# Explicitly install pandas==2.2.2 to satisfy google-colab requirements.
!pip install -q pandas==2.2.2

# Now install all other necessary libraries. numpy will be resolved by pip for other packages.
!pip install -q \
    efficientnet-pytorch \
    facenet-pytorch \
    "albumentations==1.3.1" \
    timm transformers scipy scikit-learn tqdm \
    opencv-python-headless matplotlib seaborn pyyaml

print("Cai dat xong!")


Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 102.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 105.1 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── DATASET (read-only — thu muc duoc share) ──────────────────────────────────
_candidates = [
    "/content/drive/MyDrive/FaceForensics++_C23",
    "/content/drive/Shareddrives/FaceForensics++_C23",
    "/content/drive/My Drive/FaceForensics++_C23",
]
DATASET_ROOT = None
for _p in _candidates:
    if os.path.exists(_p):
        DATASET_ROOT = _p
        break
if DATASET_ROOT is None:
    DATASET_ROOT = "/content/drive/MyDrive/FaceForensics++_C23"  # SUA NEU CAN
    print(f"[WARN] Khong tu dong tim duoc dataset, dung: {DATASET_ROOT}")
else:
    print(f"[OK] Dataset (read-only): {DATASET_ROOT}")

CSV_DIR = os.path.join(DATASET_ROOT, "csv")

# ── WORK DIR (co the ghi — MyDrive cua ban) ────────────────────────────────────
WORK_DIR       = "/content/drive/MyDrive/deepfake_work"
EXTRACTED_ROOT = os.path.join(WORK_DIR, "extracted_faces")
MANIFEST_PATH  = os.path.join(WORK_DIR, "splits", "ffpp_c23_72_14_14.json")
OUTPUT_DIR     = os.path.join(WORK_DIR, "train_outputs")

FRAMES_PER_VIDEO = 32
FAKE_CLASSES = [
    "DeepFakeDetection", "Deepfakes", "Face2Face",
    "FaceShifter", "FaceSwap", "NeuralTextures",
]
REAL_CLASS = "original"

# Tao thu muc ghi duoc
os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)
os.makedirs(EXTRACTED_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nDATA  (read-only) : {DATASET_ROOT}")
print(f"WORK  (writable)  : {WORK_DIR}")
print(f"MANIFEST          : {MANIFEST_PATH}")
print(f"EXTRACTED         : {EXTRACTED_ROOT}")
print(f"OUTPUT            : {OUTPUT_DIR}")
print(f"\nKiem tra video sources:")
for cls in [REAL_CLASS] + FAKE_CLASSES:
    p = os.path.join(DATASET_ROOT, cls)
    if os.path.exists(p):
        n = len([f for f in os.listdir(p) if f.endswith(".mp4")])
        print(f"  [OK  {n:4d} videos] {cls}")
    else:
        print(f"  [MISSING       ] {cls}")

Mounted at /content/drive
[OK] Dataset (read-only): /content/drive/MyDrive/FaceForensics++_C23

DATA  (read-only) : /content/drive/MyDrive/FaceForensics++_C23
WORK  (writable)  : /content/drive/MyDrive/deepfake_work
MANIFEST          : /content/drive/MyDrive/deepfake_work/splits/ffpp_c23_72_14_14.json
EXTRACTED         : /content/drive/MyDrive/deepfake_work/extracted_faces
OUTPUT            : /content/drive/MyDrive/deepfake_work/train_outputs

Kiem tra video sources:
  [OK  1000 videos] original
  [OK  1000 videos] DeepFakeDetection
  [OK  1000 videos] Deepfakes
  [OK  1000 videos] Face2Face
  [OK  1000 videos] FaceShifter
  [OK  1000 videos] FaceSwap
  [OK  1000 videos] NeuralTextures


In [5]:
import os, sys

# Ghi toan bo code du an vao /content/
# (Tu dong, khong can upload ZIP hay clone git)

FILES = {}

os.makedirs('/content/deepfake_detector', exist_ok=True)
open('/content/deepfake_detector/__init__.py', 'w', encoding='utf-8').write('"""\nDeepFake Detection using EfficientNet\nA robust, production-ready deepfake detection framework.\n"""\n\n__version__ = "2.0.0"\n__author__ = "Umit Kacar"\n__license__ = "MIT"\n\nfrom deepfake_detector.models import DeepFakeDetector\nfrom deepfake_detector.data import DeepFakeDataset\nfrom deepfake_detector.utils import setup_logger, calculate_metrics\n\n__all__ = [\n    "DeepFakeDetector",\n    "DeepFakeDataset",\n    "setup_logger",\n    "calculate_metrics",\n]\n')
print(f'  Ghi: /content/deepfake_detector/__init__.py')

os.makedirs('/content/deepfake_detector/models', exist_ok=True)
open('/content/deepfake_detector/models/__init__.py', 'w', encoding='utf-8').write('"""Model definitions for deepfake detection."""\n\nfrom deepfake_detector.models.efficientnet import DeepFakeDetector\nfrom deepfake_detector.models.multistream import TriStreamDeepFakeDetector\n\n__all__ = ["DeepFakeDetector", "TriStreamDeepFakeDetector"]\n')
print(f'  Ghi: /content/deepfake_detector/models/__init__.py')

os.makedirs('/content/deepfake_detector/models', exist_ok=True)
open('/content/deepfake_detector/models/efficientnet.py', 'w', encoding='utf-8').write('"""\nEfficientNet-based DeepFake Detection Model\nProvides a robust and efficient architecture for binary classification.\n"""\n\nimport torch\nimport torch.nn as nn\nfrom efficientnet_pytorch import EfficientNet\nfrom typing import Optional, Tuple\nimport logging\n\nlogger = logging.getLogger(__name__)\n\n\nclass DeepFakeDetector(nn.Module):\n    """\n    EfficientNet-based deepfake detector.\n\n    This model uses a pre-trained EfficientNet backbone with a custom\n    classification head for binary deepfake detection.\n\n    Args:\n        model_name: EfficientNet variant (e.g., \'efficientnet-b0\', \'efficientnet-b1\')\n        num_classes: Number of output classes (default: 2 for binary classification)\n        dropout_rate: Dropout probability in classification head (default: 0.5)\n        pretrained: Whether to load ImageNet pre-trained weights (default: True)\n\n    Example:\n        >>> model = DeepFakeDetector(\'efficientnet-b1\')\n        >>> output = model(torch.randn(4, 3, 224, 224))\n        >>> print(output.shape)  # torch.Size([4, 2])\n    """\n\n    def __init__(\n        self,\n        model_name: str = \'efficientnet-b1\',\n        num_classes: int = 2,\n        dropout_rate: float = 0.5,\n        pretrained: bool = True\n    ):\n        super(DeepFakeDetector, self).__init__()\n\n        self.model_name = model_name\n        self.num_classes = num_classes\n\n        # Load pre-trained EfficientNet\n        if pretrained:\n            logger.info(f"Loading pre-trained {model_name}")\n            self.backbone = EfficientNet.from_pretrained(model_name)\n        else:\n            logger.info(f"Initializing {model_name} from scratch")\n            self.backbone = EfficientNet.from_name(model_name)\n\n        # Get number of features from backbone\n        num_ftrs = self.backbone._fc.in_features\n\n        # Custom classification head\n        self.backbone._fc = nn.Sequential(\n            nn.Linear(num_ftrs, 1000, bias=True),\n            nn.ReLU(inplace=True),\n            nn.Dropout(p=dropout_rate),\n            nn.Linear(1000, num_classes, bias=True)\n        )\n\n        logger.info(f"Model initialized with {num_ftrs} features -> {num_classes} classes")\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        """\n        Forward pass through the network.\n\n        Args:\n            x: Input tensor of shape (batch_size, 3, height, width)\n\n        Returns:\n            Logits tensor of shape (batch_size, num_classes)\n        """\n        return self.backbone(x)\n\n    def get_image_size(self) -> int:\n        """Get the expected input image size for this model variant."""\n        return EfficientNet.get_image_size(self.model_name)\n\n    def freeze_backbone(self, freeze: bool = True) -> None:\n        """\n        Freeze or unfreeze the backbone parameters.\n\n        Args:\n            freeze: If True, freeze backbone; if False, unfreeze\n        """\n        for param in self.backbone.parameters():\n            param.requires_grad = not freeze\n\n        # Always keep classification head trainable\n        for param in self.backbone._fc.parameters():\n            param.requires_grad = True\n\n        status = "frozen" if freeze else "unfrozen"\n        logger.info(f"Backbone {status}, classification head trainable")\n\n    def load_checkpoint(self, checkpoint_path: str, device: str = \'cpu\') -> None:\n        """\n        Load model weights from checkpoint.\n\n        Args:\n            checkpoint_path: Path to checkpoint file\n            device: Device to load the model on\n        """\n        logger.info(f"Loading checkpoint from {checkpoint_path}")\n        checkpoint = torch.load(checkpoint_path, map_location=device)\n\n        if \'model_state_dict\' in checkpoint:\n            self.load_state_dict(checkpoint[\'model_state_dict\'])\n            logger.info(f"Loaded model from epoch {checkpoint.get(\'epoch\', \'unknown\')}")\n            return checkpoint\n        else:\n            self.load_state_dict(checkpoint)\n            logger.info("Loaded model weights")\n            return {}\n\n    def save_checkpoint(\n        self,\n        checkpoint_path: str,\n        epoch: Optional[int] = None,\n        optimizer_state: Optional[dict] = None,\n        scheduler_state: Optional[dict] = None,\n        metrics: Optional[dict] = None\n    ) -> None:\n        """\n        Save model checkpoint.\n\n        Args:\n            checkpoint_path: Path to save checkpoint\n            epoch: Current epoch number\n            optimizer_state: Optimizer state dict\n            metrics: Dictionary of metrics to save\n        """\n        checkpoint = {\n            \'model_state_dict\': self.state_dict(),\n            \'model_name\': self.model_name,\n            \'num_classes\': self.num_classes,\n        }\n\n        if epoch is not None:\n            checkpoint[\'epoch\'] = epoch\n        if optimizer_state is not None:\n            checkpoint[\'optimizer_state_dict\'] = optimizer_state\n        if scheduler_state is not None:\n            checkpoint[\'scheduler_state_dict\'] = scheduler_state\n        if metrics is not None:\n            checkpoint[\'metrics\'] = metrics\n\n        torch.save(checkpoint, checkpoint_path)\n        logger.info(f"Checkpoint saved to {checkpoint_path}")\n\n    def count_parameters(self) -> Tuple[int, int]:\n        """\n        Count total and trainable parameters.\n\n        Returns:\n            Tuple of (total_params, trainable_params)\n        """\n        total_params = sum(p.numel() for p in self.parameters())\n        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)\n        return total_params, trainable_params\n\n    def __repr__(self) -> str:\n        total, trainable = self.count_parameters()\n        return (\n            f"DeepFakeDetector(\\n"\n            f"  model={self.model_name},\\n"\n            f"  num_classes={self.num_classes},\\n"\n            f"  total_params={total:,},\\n"\n            f"  trainable_params={trainable:,}\\n"\n            f")"\n        )\n')
print(f'  Ghi: /content/deepfake_detector/models/efficientnet.py')

os.makedirs('/content/deepfake_detector/models', exist_ok=True)
open('/content/deepfake_detector/models/multistream.py', 'w', encoding='utf-8').write('"""\nTri-stream deepfake detector — SVG feature-vector pipeline.\n\nArchitecture (matching feature_vector_pipeline.svg):\n\n  Input (H×W×3, ImageNet-normalised)\n    ├─ Stream 1 – RGB:       original → EfficientNet-B4 (encoder only) → f_rgb  [B, D]\n    ├─ Stream 2 – Frequency: FFT magnitude + phase + DCT → EfficientNet-B4 → f_freq [B, D]\n    └─ Stream 3 – SRM:       30 high-pass filters → 3-ch → EfficientNet-B4 → f_srm  [B, D]\n                                       ↓\n             Channel Attention Fusion  (w_rgb·f_rgb + w_freq·f_freq + w_srm·f_srm)\n                                       ↓\n                            fused vector [B, D]\n                                       ↓\n              LayerNorm → Dropout(0.5) → FC(512) → GELU → FC(num_classes)\n\nKey differences from the old logit-fusion implementation:\n- All three branches share the same EfficientNet variant (default: B4).\n- Branches act as *feature extractors* — the per-branch FC head is removed.\n- Fusion happens at the 1792-d feature level (Channel Attention), not over 2-class logits.\n- A single shared classifier head produces the final prediction.\n- Frequency stream now encodes FFT magnitude + phase + DCT (3 complementary channels).\n- SRM stream: 30 learnable Laplacian-like high-pass kernels → compressed to 3 channels.\n\nBackward compatibility:\n- The constructor accepts the old `freq_model`, `srm_model`, `freq_band_r1`, `freq_band_r2`\n  arguments but ignores them (they are superseded by the shared `rgb_model` backbone).\n- Output is still [B, num_classes] logits — `CrossEntropyLoss` in `train.py` is unchanged.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom efficientnet_pytorch import EfficientNet\n\n\n# ──────────────────────────────────────────────────────────────────────────────\n# Helpers\n# ──────────────────────────────────────────────────────────────────────────────\n\ndef _effnet_input_size(model_name: str) -> int:\n    sizes = {"b0": 224, "b1": 240, "b2": 260, "b3": 300,\n             "b4": 380, "b5": 456, "b6": 528, "b7": 600}\n    name = model_name.lower()\n    for k, v in sizes.items():\n        if k in name:\n            return v\n    return 224\n\n\ndef _effnet_feat_dim(model_name: str) -> int:\n    """Feature dimension (before FC head) for each EfficientNet variant."""\n    dims = {"b0": 1280, "b1": 1280, "b2": 1408, "b3": 1536,\n            "b4": 1792, "b5": 2048, "b6": 2304, "b7": 2560}\n    name = model_name.lower()\n    for k, v in dims.items():\n        if k in name:\n            return v\n    return 1280\n\n\n# ──────────────────────────────────────────────────────────────────────────────\n# Channel Attention Fusion\n# ──────────────────────────────────────────────────────────────────────────────\n\nclass ChannelAttentionFusion(nn.Module):\n    """\n    Learns soft attention weights over N feature vectors and returns their\n    weighted sum.\n\n    w = softmax( MLP( concat(f_1, …, f_N) ) )   shape [B, N]\n    out = sum_i  w_i * f_i                        shape [B, feat_dim]\n    """\n\n    def __init__(self, feat_dim: int, num_streams: int = 3, reduction: int = 16):\n        super().__init__()\n        hidden = max(feat_dim // reduction, 32)\n        self.attn = nn.Sequential(\n            nn.Linear(feat_dim * num_streams, hidden),\n            nn.ReLU(inplace=True),\n            nn.Linear(hidden, num_streams),\n        )\n\n    def forward(self, feats: List[torch.Tensor]) -> torch.Tensor:\n        cat = torch.cat(feats, dim=1)                        # [B, D*N]\n        w = F.softmax(self.attn(cat), dim=1)                 # [B, N]\n        return sum(w[:, i : i + 1] * f for i, f in enumerate(feats))  # [B, D]\n\n\n# ──────────────────────────────────────────────────────────────────────────────\n# Main detector\n# ──────────────────────────────────────────────────────────────────────────────\n\nclass TriStreamDeepFakeDetector(nn.Module):\n    """\n    Tri-stream deepfake detector matching the SVG feature-vector pipeline.\n\n    Args:\n        rgb_model:   EfficientNet variant used as the shared backbone for all\n                     three streams (e.g. \'efficientnet-b4\').\n        freq_model:  Deprecated — kept for backward compatibility, ignored.\n        srm_model:   Deprecated — kept for backward compatibility, ignored.\n        srm_filters: Number of learnable high-pass convolutional filters for\n                     the SRM stream (default: 30).\n        freq_band_r1, freq_band_r2: Deprecated — ignored.\n        pretrained:  Load ImageNet pre-trained weights for the backbones.\n        num_classes: Output classes (default: 2 for binary classification).\n    """\n\n    def __init__(\n        self,\n        rgb_model: str = "efficientnet-b4",\n        freq_model: str = "efficientnet-b4",   # deprecated — ignored\n        srm_model: str = "efficientnet-b4",    # deprecated — ignored\n        srm_filters: int = 30,\n        freq_band_r1: float = 0.33,            # deprecated — ignored\n        freq_band_r2: float = 0.66,            # deprecated — ignored\n        pretrained: bool = True,\n        num_classes: int = 2,\n    ):\n        super().__init__()\n\n        # All three streams use the same backbone variant.\n        backbone = rgb_model\n        self._backbone_name = backbone\n        self._input_size = _effnet_input_size(backbone)\n        self._feat_dim = _effnet_feat_dim(backbone)\n\n        # ── Three feature encoders (EfficientNet without FC head) ────────────\n        self.rgb_encoder = self._build_encoder(backbone, pretrained)\n        self.freq_encoder = self._build_encoder(backbone, pretrained)\n        self.srm_encoder = self._build_encoder(backbone, pretrained)\n\n        # ── SRM block: learnable high-pass filters → 3 channels ─────────────\n        # Conv(1 → srm_filters, 5×5)  then  Conv(srm_filters → 3, 1×1)\n        self._srm_conv = nn.Conv2d(1, srm_filters, kernel_size=5, padding=2, bias=False)\n        self._srm_to3 = nn.Conv2d(srm_filters, 3, kernel_size=1, bias=True)\n        self._init_srm_kernels()\n\n        # ── Channel Attention Fusion ─────────────────────────────────────────\n        self.fusion = ChannelAttentionFusion(\n            feat_dim=self._feat_dim, num_streams=3, reduction=16\n        )\n\n        # ── Classifier head ──────────────────────────────────────────────────\n        # LayerNorm → Dropout → FC(512) → GELU → FC(num_classes)\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(self._feat_dim),\n            nn.Dropout(0.5),\n            nn.Linear(self._feat_dim, 512),\n            nn.GELU(),\n            nn.Linear(512, num_classes),\n        )\n\n        # ── ImageNet normalisation constants (registered as buffers) ─────────\n        mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)\n        std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)\n        self.register_buffer("_mean", mean)\n        self.register_buffer("_std", std)\n\n    # ── Construction helpers ──────────────────────────────────────────────────\n\n    @staticmethod\n    def _build_encoder(model_name: str, pretrained: bool) -> EfficientNet:\n        """Return a bare EfficientNet (pre-trained weights, FC head unused)."""\n        if pretrained:\n            return EfficientNet.from_pretrained(model_name)\n        return EfficientNet.from_name(model_name)\n\n    def _init_srm_kernels(self) -> None:\n        """Initialise SRM conv weights with Laplacian-like kernels + small noise."""\n        base = torch.tensor(\n            [[0.0,  0.0, -1.0,  0.0,  0.0],\n             [0.0,  0.0, -2.0,  0.0,  0.0],\n             [-1.0, -2.0, 16.0, -2.0, -1.0],\n             [0.0,  0.0, -2.0,  0.0,  0.0],\n             [0.0,  0.0, -1.0,  0.0,  0.0]],\n            dtype=torch.float32,\n        )\n        base = base / (base.abs().sum() + 1e-6)\n        with torch.no_grad():\n            w = self._srm_conv.weight  # [K, 1, 5, 5]\n            for k in range(w.shape[0]):\n                w[k, 0].copy_(base + 0.02 * torch.randn_like(base))\n\n    # ── ImageNet normalisation helpers ────────────────────────────────────────\n\n    def _denorm(self, x: torch.Tensor) -> torch.Tensor:\n        """Undo ImageNet normalisation → [0, 1]."""\n        return (x * self._std + self._mean).clamp(0.0, 1.0)\n\n    def _renorm(self, x01: torch.Tensor) -> torch.Tensor:\n        """Apply ImageNet normalisation to a [0, 1] tensor."""\n        return (x01 - self._mean) / (self._std + 1e-6)\n\n    @staticmethod\n    def _scale01(t: torch.Tensor) -> torch.Tensor:\n        """Min-max scale each sample to [0, 1] over its spatial dimensions."""\n        mn = t.amin(dim=(-2, -1), keepdim=True)\n        mx = t.amax(dim=(-2, -1), keepdim=True)\n        return (t - mn) / (mx - mn + 1e-6)\n\n    # ── Feature extraction ────────────────────────────────────────────────────\n\n    def _encode(self, encoder: EfficientNet, x: torch.Tensor) -> torch.Tensor:\n        """\n        Forward through EfficientNet up to global average-pool; skip the FC\n        head.  Returns a feature vector of shape [B, feat_dim].\n        """\n        x = encoder.extract_features(x)   # [B, C, H\', W\']\n        x = encoder._avg_pooling(x)        # [B, C, 1, 1]\n        x = x.flatten(start_dim=1)         # [B, C]\n        x = encoder._dropout(x)\n        return x\n\n    # ── Frequency stream ──────────────────────────────────────────────────────\n\n    @staticmethod\n    def _dct2d(x: torch.Tensor) -> torch.Tensor:\n        """\n        2-D DCT-II via FFT applied row-wise then column-wise.\n\n        Input : x  [B, H, W]  (real-valued, any range)\n        Output: [B, H, W]  DCT coefficients\n        """\n        def _dct1d(v: torch.Tensor, dim: int) -> torch.Tensor:\n            N = v.shape[dim]\n            # Mirror-extend so FFT gives DCT-II.\n            v_ext = torch.cat([v, v.flip(dim)], dim=dim)\n            V = torch.fft.rfft(v_ext, dim=dim)\n            # Phase shift.\n            k_shape = [1] * v.ndim\n            k_shape[dim] = N\n            k = torch.arange(N, device=v.device, dtype=torch.float32).view(k_shape)\n            phase = torch.exp(\n                -1j * torch.pi * k.to(torch.complex64) / (2 * N)\n            )\n            idx = [slice(None)] * v.ndim\n            idx[dim] = slice(0, N)\n            return (V[tuple(idx)] * phase).real\n\n        return _dct1d(_dct1d(x, dim=-1), dim=-2)\n\n    def _compute_freq_channels(self, x_rgb: torch.Tensor) -> torch.Tensor:\n        """\n        Derive 3-channel frequency representation from the RGB input:\n          Ch 1 – FFT log-magnitude spectrum   (captures GAN frequency artefacts)\n          Ch 2 – FFT phase spectrum            (captures geometric manipulations)\n          Ch 3 – DCT log-magnitude             (complementary energy distribution)\n\n        Returns an ImageNet-normalised tensor [B, 3, input_size, input_size].\n        """\n        x01 = self._denorm(x_rgb)\n        # Luminance grayscale.\n        gray = (0.2989 * x01[:, 0:1]\n                + 0.5870 * x01[:, 1:2]\n                + 0.1140 * x01[:, 2:3])           # [B, 1, H, W]\n        gray = F.interpolate(\n            gray, size=(self._input_size, self._input_size),\n            mode="bilinear", align_corners=False\n        )\n        g = gray.squeeze(1)                        # [B, H, W]\n\n        # FFT (centred).\n        fft_s = torch.fft.fftshift(torch.fft.fft2(g), dim=(-2, -1))\n\n        # Ch 1: log-magnitude.\n        mag = self._scale01(torch.log1p(torch.abs(fft_s)))\n\n        # Ch 2: phase → [0, 1].\n        phase = (torch.angle(fft_s) / torch.pi + 1.0) * 0.5\n\n        # Ch 3: DCT log-magnitude.\n        dct = self._scale01(torch.log1p(self._dct2d(g).abs()))\n\n        freq01 = torch.stack([mag, phase, dct], dim=1)   # [B, 3, H, W]\n        return self._renorm(freq01)\n\n    # ── SRM stream ────────────────────────────────────────────────────────────\n\n    def _compute_srm_channels(self, x_rgb: torch.Tensor) -> torch.Tensor:\n        """\n        Apply 30 learnable Laplacian-like high-pass filters to grayscale image,\n        then compress to 3 channels.\n\n        Returns an ImageNet-normalised tensor [B, 3, input_size, input_size].\n        """\n        x01 = self._denorm(x_rgb)\n        gray = (0.2989 * x01[:, 0:1]\n                + 0.5870 * x01[:, 1:2]\n                + 0.1140 * x01[:, 2:3])            # [B, 1, H, W]\n        gray = F.interpolate(\n            gray, size=(self._input_size, self._input_size),\n            mode="bilinear", align_corners=False\n        )\n        noise = torch.abs(self._srm_conv(gray))    # [B, 30, H, W]\n        out3 = F.relu(self._srm_to3(noise))         # [B, 3, H, W]\n        mn = out3.amin(dim=(-2, -1), keepdim=True)\n        mx = out3.amax(dim=(-2, -1), keepdim=True)\n        out01 = (out3 - mn) / (mx - mn + 1e-6)\n        return self._renorm(out01)\n\n    # ── Forward pass ─────────────────────────────────────────────────────────\n\n    def forward(self, x_rgb: torch.Tensor) -> torch.Tensor:\n        """\n        Args:\n            x_rgb: ImageNet-normalised face crop [B, 3, H, W].\n\n        Returns:\n            Class logits [B, num_classes].\n        """\n        # Resize input to backbone\'s expected resolution if necessary.\n        if x_rgb.shape[-1] != self._input_size or x_rgb.shape[-2] != self._input_size:\n            x_rgb = F.interpolate(\n                x_rgb, size=(self._input_size, self._input_size),\n                mode="bilinear", align_corners=False,\n            )\n\n        # ── 3 feature vectors ────────────────────────────────────────────────\n        f_rgb = self._encode(self.rgb_encoder, x_rgb)\n\n        x_freq = self._compute_freq_channels(x_rgb)\n        f_freq = self._encode(self.freq_encoder, x_freq)\n\n        x_srm = self._compute_srm_channels(x_rgb)\n        f_srm = self._encode(self.srm_encoder, x_srm)\n\n        # ── Channel Attention Fusion → fused vector ──────────────────────────\n        fused = self.fusion([f_rgb, f_freq, f_srm])   # [B, feat_dim]\n\n        # ── Classify ─────────────────────────────────────────────────────────\n        return self.classifier(fused)                  # [B, num_classes]\n\n    # ── Utility methods ───────────────────────────────────────────────────────\n\n    def count_parameters(self) -> Tuple[int, int]:\n        total = sum(p.numel() for p in self.parameters())\n        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)\n        return total, trainable\n\n    def save_checkpoint(\n        self,\n        checkpoint_path: str,\n        epoch: Optional[int] = None,\n        optimizer_state: Optional[dict] = None,\n        scheduler_state: Optional[dict] = None,\n        metrics: Optional[dict] = None,\n    ) -> None:\n        ckpt: Dict[str, Any] = {\n            "model_state_dict": self.state_dict(),\n            "backbone": self._backbone_name,\n            "num_classes": self.classifier[-1].out_features,\n        }\n        if epoch is not None:\n            ckpt["epoch"] = epoch\n        if optimizer_state is not None:\n            ckpt["optimizer_state_dict"] = optimizer_state\n        if scheduler_state is not None:\n            ckpt["scheduler_state_dict"] = scheduler_state\n        if metrics is not None:\n            ckpt["metrics"] = metrics\n        torch.save(ckpt, checkpoint_path)\n\n    def load_checkpoint(self, checkpoint_path: str, device: str = "cpu") -> dict:\n        ckpt = torch.load(checkpoint_path, map_location=device)\n        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:\n            self.load_state_dict(ckpt["model_state_dict"])\n            return ckpt\n        self.load_state_dict(ckpt)\n        return {}\n')
print(f'  Ghi: /content/deepfake_detector/models/multistream.py')

os.makedirs('/content/deepfake_detector/data', exist_ok=True)
open('/content/deepfake_detector/data/__init__.py', 'w', encoding='utf-8').write('"""Data loading and preprocessing modules."""\n\nfrom deepfake_detector.data.dataset import DeepFakeDataset\nfrom deepfake_detector.data.dataset import create_combined_dataset\nfrom deepfake_detector.data.transforms import get_train_transforms, get_val_transforms\nfrom deepfake_detector.data.loader import create_dataloaders\n\n__all__ = [\n    "DeepFakeDataset",\n    "create_combined_dataset",\n    "get_train_transforms",\n    "get_val_transforms",\n    "create_dataloaders",\n]\n')
print(f'  Ghi: /content/deepfake_detector/data/__init__.py')

os.makedirs('/content/deepfake_detector/data', exist_ok=True)
open('/content/deepfake_detector/data/dataset.py', 'w', encoding='utf-8').write('"""\nDataset class for DeepFake Detection\nHandles loading and preprocessing of facial images.\n"""\n\nimport os\nimport glob\nfrom typing import List, Tuple, Optional, Callable\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\nfrom albumentations import Compose\nimport logging\n\nlogger = logging.getLogger(__name__)\n\n\nclass DeepFakeDataset(Dataset):\n    """\n    PyTorch Dataset for DeepFake Detection.\n\n    Supports multiple data directories and balanced sampling.\n\n    Args:\n        data_config: List of tuples [(directory_path, num_samples), ...]\n        is_real: Whether this dataset contains real (True) or fake (False) images\n        transform: Albumentations transform composition\n        image_extensions: Tuple of supported image extensions\n\n    Example:\n        >>> config = [("/path/to/real", 1000), ("/path/to/real2", 500)]\n        >>> dataset = DeepFakeDataset(config, is_real=True, transform=transforms)\n        >>> image, label = dataset[0]\n    """\n\n    def __init__(\n        self,\n        data_config: List[Tuple[str, int]],\n        is_real: bool = True,\n        transform: Optional[Compose] = None,\n        image_extensions: Tuple[str, ...] = (\'.jpg\', \'.jpeg\', \'.png\')\n    ):\n        super().__init__()\n\n        self.data_config = data_config\n        self.is_real = is_real\n        self.transform = transform\n        self.image_extensions = image_extensions\n\n        # Build dataset\n        self.data = self._build_dataset()\n\n        logger.info(\n            f"{\'Real\' if is_real else \'Fake\'} dataset initialized: "\n            f"{len(self.data)} samples"\n        )\n\n    def _build_dataset(self) -> pd.DataFrame:\n        """Build dataset from configuration."""\n        all_data = []\n\n        for directory, sample_num in self.data_config:\n            if not os.path.exists(directory):\n                logger.warning(f"Directory not found: {directory}, skipping...")\n                continue\n\n            # Collect all image paths\n            image_paths = []\n            for ext in self.image_extensions:\n                image_paths.extend(glob.glob(os.path.join(directory, f\'*{ext}\')))\n\n            if len(image_paths) == 0:\n                logger.warning(f"No images found in {directory}")\n                continue\n\n            # Create dataframe\n            df = pd.DataFrame(image_paths, columns=[\'image_path\'])\n            df[\'real\'] = 1.0 if self.is_real else 0.0\n            df[\'fake\'] = 0.0 if self.is_real else 1.0\n\n            # Sample if necessary\n            if sample_num > 0 and len(df) >= sample_num:\n                df = df.sample(n=sample_num, random_state=42, replace=False)\n            elif sample_num > len(df):\n                logger.warning(\n                    f"Requested {sample_num} samples but only {len(df)} available in {directory}"\n                )\n\n            all_data.append(df)\n\n            logger.debug(f"Loaded {len(df)} samples from {directory}")\n\n        if not all_data:\n            raise ValueError("No valid data found in any directory")\n\n        return pd.concat(all_data, ignore_index=True)\n\n    def __len__(self) -> int:\n        """Return dataset size."""\n        return len(self.data)\n\n    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:\n        """\n        Get item by index.\n\n        Args:\n            idx: Index of the sample\n\n        Returns:\n            Tuple of (image_tensor, label_tensor)\n        """\n        # Get image path and labels\n        image_path = self.data.loc[idx, \'image_path\']\n        real_label = self.data.loc[idx, \'real\']\n        fake_label = self.data.loc[idx, \'fake\']\n\n        # Load image\n        image_bgr = cv2.imread(image_path)\n\n        if image_bgr is None:\n            logger.error(f"Failed to load image: {image_path}")\n            # Return a black image as fallback\n            image = np.zeros((224, 224, 3), dtype=np.uint8)\n        else:\n            # Convert BGR to RGB\n            image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)\n\n        # Apply transforms\n        if self.transform is not None:\n            transformed = self.transform(image=image)\n            image = transformed[\'image\']\n\n        # Create label (class index)\n        # Convention used across this repo:\n        #   1 = REAL (bona-fide)\n        #   0 = FAKE (deepfake)\n        label = torch.tensor(int(real_label), dtype=torch.long)\n\n        return image, label\n\n    def get_labels(self) -> np.ndarray:\n        """Get all labels as numpy array."""\n        return self.data[\'real\'].values.astype(int)\n\n    def get_class_weights(self) -> torch.Tensor:\n        """\n        Calculate class weights for balanced training.\n\n        Returns:\n            Tensor of class weights\n        """\n        labels = self.get_labels()\n        class_counts = np.bincount(labels)\n        total = len(labels)\n        weights = total / (len(class_counts) * class_counts)\n        return torch.tensor(weights, dtype=torch.float32)\n\n\nclass CombinedDataset(Dataset):\n    """\n    Combined real+fake dataset.\n\n    Defined at module scope so it is picklable by multiprocessing DataLoader\n    workers on Windows.\n    """\n\n    def __init__(\n        self,\n        data: pd.DataFrame,\n        transform: Optional[Compose] = None,\n        return_paths: bool = False,\n    ):\n        self.data = data\n        self.transform = transform\n        self.return_paths = return_paths\n\n    def __len__(self) -> int:\n        return len(self.data)\n\n    def __getitem__(self, idx: int):\n        image_path = self.data.loc[idx, \'image_path\']\n        real_label = self.data.loc[idx, \'real\']\n\n        image_bgr = cv2.imread(image_path)\n        if image_bgr is None:\n            image = np.zeros((224, 224, 3), dtype=np.uint8)\n        else:\n            image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)\n\n        if self.transform is not None:\n            transformed = self.transform(image=image)\n            image = transformed[\'image\']\n\n        # Convention used across this repo:\n        #   1 = REAL (bona-fide)\n        #   0 = FAKE (deepfake)\n        label = torch.tensor(int(real_label), dtype=torch.long)\n        if self.return_paths:\n            return image, label, image_path\n        return image, label\n\n\ndef create_combined_dataset(\n    real_config: List[Tuple[str, int]],\n    fake_config: List[Tuple[str, int]],\n    transform: Optional[Compose] = None,\n    return_paths: bool = False,\n) -> Dataset:\n    """\n    Create a combined dataset with both real and fake samples.\n\n    Args:\n        real_config: Configuration for real images\n        fake_config: Configuration for fake images\n        transform: Transform to apply\n\n    Returns:\n        Combined dataset\n\n    Example:\n        >>> real_cfg = [("/path/real", 1000)]\n        >>> fake_cfg = [("/path/fake", 1000)]\n        >>> dataset = create_combined_dataset(real_cfg, fake_cfg, transforms)\n    """\n    real_dataset = DeepFakeDataset(real_config, is_real=True, transform=transform)\n    fake_dataset = DeepFakeDataset(fake_config, is_real=False, transform=transform)\n\n    # Combine datasets\n    combined_data = pd.concat(\n        [real_dataset.data, fake_dataset.data],\n        ignore_index=True\n    )\n\n    return CombinedDataset(combined_data, transform, return_paths=return_paths)\n')
print(f'  Ghi: /content/deepfake_detector/data/dataset.py')

os.makedirs('/content/deepfake_detector/data', exist_ok=True)
open('/content/deepfake_detector/data/transforms.py', 'w', encoding='utf-8').write('"""\nData augmentation and preprocessing transforms.\nUses albumentations for efficient image transformations.\n"""\n\nfrom albumentations import (\n    Compose, HorizontalFlip, RandomResizedCrop, Resize,\n    Normalize, VerticalFlip, Rotate, ShiftScaleRotate,\n    OpticalDistortion, GridDistortion, ElasticTransform,\n    JpegCompression, HueSaturationValue, RGBShift,\n    RandomBrightness, RandomContrast, Blur, MotionBlur,\n    MedianBlur, GaussNoise, CLAHE, RandomGamma, CoarseDropout\n)\nfrom albumentations.pytorch import ToTensorV2\nfrom typing import Optional\n\n\ndef get_train_transforms(\n    image_size: int = 224,\n    use_heavy_augmentation: bool = False\n) -> Compose:\n    """\n    Get training data augmentation pipeline.\n\n    Args:\n        image_size: Target image size\n        use_heavy_augmentation: Whether to use aggressive augmentation\n\n    Returns:\n        Albumentations Compose object with training transforms\n\n    Example:\n        >>> transforms = get_train_transforms(224, use_heavy_augmentation=True)\n        >>> augmented = transforms(image=image)\n    """\n    if use_heavy_augmentation:\n        return Compose([\n            # Geometric transforms\n            HorizontalFlip(p=0.5),\n            VerticalFlip(p=0.1),\n            RandomResizedCrop(\n                image_size, image_size,\n                scale=(0.5, 1.0),\n                ratio=(0.9, 1.1),\n                p=0.5\n            ),\n            Rotate(limit=15, p=0.3),\n            ShiftScaleRotate(\n                shift_limit=0.1,\n                scale_limit=0.1,\n                rotate_limit=15,\n                p=0.3\n            ),\n\n            # Optical distortions\n            OpticalDistortion(distort_limit=0.1, p=0.2),\n            GridDistortion(p=0.2),\n            ElasticTransform(\n                alpha=1,\n                sigma=50,\n                alpha_affine=50,\n                p=0.2\n            ),\n\n            # Compression and noise\n            JpegCompression(quality_lower=75, quality_upper=100, p=0.3),\n            GaussNoise(var_limit=(10.0, 50.0), p=0.3),\n\n            # Color augmentations\n            HueSaturationValue(\n                hue_shift_limit=20,\n                sat_shift_limit=30,\n                val_shift_limit=20,\n                p=0.3\n            ),\n            RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.3),\n            RandomBrightness(limit=0.2, p=0.3),\n            RandomContrast(limit=0.2, p=0.3),\n            RandomGamma(gamma_limit=(80, 120), p=0.3),\n            CLAHE(p=0.2),\n\n            # Blur\n            Blur(blur_limit=3, p=0.2),\n            MotionBlur(blur_limit=3, p=0.2),\n            MedianBlur(blur_limit=3, p=0.2),\n\n            # Cutout\n            CoarseDropout(\n                max_holes=8,\n                max_height=32,\n                max_width=32,\n                fill_value=0,\n                p=0.3\n            ),\n\n            # Final resize and normalization\n            Resize(image_size, image_size, always_apply=True),\n            Normalize(\n                mean=[0.485, 0.456, 0.406],\n                std=[0.229, 0.224, 0.225],\n                always_apply=True\n            ),\n            ToTensorV2()\n        ])\n    else:\n        return Compose([\n            HorizontalFlip(p=0.5),\n            RandomResizedCrop(\n                image_size, image_size,\n                scale=(0.5, 1.0),\n                p=0.5\n            ),\n            Resize(image_size, image_size, always_apply=True),\n            Normalize(\n                mean=[0.485, 0.456, 0.406],\n                std=[0.229, 0.224, 0.225],\n                always_apply=True\n            ),\n            ToTensorV2()\n        ])\n\n\ndef get_val_transforms(image_size: int = 224) -> Compose:\n    """\n    Get validation/test data preprocessing pipeline.\n\n    Args:\n        image_size: Target image size\n\n    Returns:\n        Albumentations Compose object with validation transforms\n\n    Example:\n        >>> transforms = get_val_transforms(224)\n        >>> preprocessed = transforms(image=image)\n    """\n    return Compose([\n        Resize(image_size, image_size, always_apply=True),\n        Normalize(\n            mean=[0.485, 0.456, 0.406],\n            std=[0.229, 0.224, 0.225],\n            always_apply=True\n        ),\n        ToTensorV2()\n    ])\n\n\ndef get_test_time_augmentation_transforms(image_size: int = 224) -> list:\n    """\n    Get multiple transform variants for test-time augmentation.\n\n    Args:\n        image_size: Target image size\n\n    Returns:\n        List of transform compositions for TTA\n\n    Example:\n        >>> tta_transforms = get_test_time_augmentation_transforms(224)\n        >>> predictions = [model(transform(image=img)[\'image\']) for transform in tta_transforms]\n    """\n    return [\n        # Original\n        get_val_transforms(image_size),\n\n        # Horizontal flip\n        Compose([\n            HorizontalFlip(p=1.0),\n            Resize(image_size, image_size, always_apply=True),\n            Normalize(\n                mean=[0.485, 0.456, 0.406],\n                std=[0.229, 0.224, 0.225],\n                always_apply=True\n            ),\n            ToTensorV2()\n        ]),\n\n        # Slight brightness adjustment\n        Compose([\n            RandomBrightness(limit=0.1, p=1.0),\n            Resize(image_size, image_size, always_apply=True),\n            Normalize(\n                mean=[0.485, 0.456, 0.406],\n                std=[0.229, 0.224, 0.225],\n                always_apply=True\n            ),\n            ToTensorV2()\n        ]),\n\n        # Slight contrast adjustment\n        Compose([\n            RandomContrast(limit=0.1, p=1.0),\n            Resize(image_size, image_size, always_apply=True),\n            Normalize(\n                mean=[0.485, 0.456, 0.406],\n                std=[0.229, 0.224, 0.225],\n                always_apply=True\n            ),\n            ToTensorV2()\n        ]),\n    ]\n')
print(f'  Ghi: /content/deepfake_detector/data/transforms.py')

os.makedirs('/content/deepfake_detector/data', exist_ok=True)
open('/content/deepfake_detector/data/loader.py', 'w', encoding='utf-8').write('"""\nDataLoader creation and configuration.\nHandles multi-process data loading with proper worker initialization.\n"""\n\nimport torch\nfrom torch.utils.data import DataLoader, Dataset, Sampler\nfrom typing import Optional, Tuple\nimport logging\n\nlogger = logging.getLogger(__name__)\n\n\ndef create_dataloaders(\n    train_dataset: Optional[Dataset] = None,\n    val_dataset: Optional[Dataset] = None,\n    test_dataset: Optional[Dataset] = None,\n    batch_size: int = 32,\n    num_workers: int = 4,\n    pin_memory: bool = True,\n    drop_last_train: bool = True,\n    train_sampler: Optional[Sampler] = None,\n) -> Tuple[Optional[DataLoader], ...]:\n    """\n    Create PyTorch DataLoaders for train/val/test datasets.\n\n    Args:\n        train_dataset: Training dataset\n        val_dataset: Validation dataset\n        test_dataset: Test dataset\n        batch_size: Batch size for all loaders\n        num_workers: Number of workers for data loading\n        pin_memory: Whether to pin memory (faster transfer to GPU)\n        drop_last_train: Whether to drop last incomplete batch in training\n\n    Returns:\n        Tuple of (train_loader, val_loader, test_loader)\n        Returns None for datasets that are not provided\n\n    Example:\n        >>> train_loader, val_loader, _ = create_dataloaders(\n        ...     train_dataset=train_ds,\n        ...     val_dataset=val_ds,\n        ...     batch_size=64,\n        ...     num_workers=8\n        ... )\n    """\n    train_loader = None\n    val_loader = None\n    test_loader = None\n\n    if train_dataset is not None:\n        train_loader = DataLoader(\n            train_dataset,\n            batch_size=batch_size,\n            shuffle=train_sampler is None,\n            sampler=train_sampler,\n            num_workers=num_workers,\n            pin_memory=pin_memory,\n            drop_last=drop_last_train,\n            persistent_workers=num_workers > 0\n        )\n        logger.info(\n            f"Train DataLoader created: {len(train_dataset)} samples, "\n            f"{len(train_loader)} batches"\n        )\n\n    if val_dataset is not None:\n        val_loader = DataLoader(\n            val_dataset,\n            batch_size=batch_size,\n            shuffle=False,\n            num_workers=num_workers,\n            pin_memory=pin_memory,\n            drop_last=False,\n            persistent_workers=num_workers > 0\n        )\n        logger.info(\n            f"Validation DataLoader created: {len(val_dataset)} samples, "\n            f"{len(val_loader)} batches"\n        )\n\n    if test_dataset is not None:\n        test_loader = DataLoader(\n            test_dataset,\n            batch_size=batch_size,\n            shuffle=False,\n            num_workers=num_workers,\n            pin_memory=pin_memory,\n            drop_last=False,\n            persistent_workers=num_workers > 0\n        )\n        logger.info(\n            f"Test DataLoader created: {len(test_dataset)} samples, "\n            f"{len(test_loader)} batches"\n        )\n\n    return train_loader, val_loader, test_loader\n\n\ndef get_optimal_num_workers() -> int:\n    """\n    Automatically determine optimal number of workers.\n\n    Returns:\n        Recommended number of workers\n\n    Example:\n        >>> num_workers = get_optimal_num_workers()\n        >>> print(f"Using {num_workers} workers")\n    """\n    import os\n    import multiprocessing as mp\n\n    # Get number of CPUs\n    num_cpus = mp.cpu_count()\n\n    # Heuristic: use 75% of CPUs, but not more than 8\n    optimal = min(max(1, int(num_cpus * 0.75)), 8)\n\n    logger.info(f"Detected {num_cpus} CPUs, recommending {optimal} workers")\n    return optimal\n')
print(f'  Ghi: /content/deepfake_detector/data/loader.py')

os.makedirs('/content/deepfake_detector/utils', exist_ok=True)
open('/content/deepfake_detector/utils/__init__.py', 'w', encoding='utf-8').write('"""Utility functions for training, evaluation, and visualization."""\n\nfrom deepfake_detector.utils.metrics import (\n    calculate_metrics,\n    get_EER_states,\n    get_HTER_at_thr,\n    eval_state,\n    calculate_comprehensive_metrics,\n    print_metrics,\n)\nfrom deepfake_detector.utils.logger import setup_logger, get_logger\nfrom deepfake_detector.utils.visualization import (\n    plot_confusion_matrix,\n    plot_roc_curve,\n    plot_training_history\n)\n\n__all__ = [\n    "calculate_metrics",\n    "get_EER_states",\n    "get_HTER_at_thr",\n    "eval_state",\n    "calculate_comprehensive_metrics",\n    "print_metrics",\n    "setup_logger",\n    "get_logger",\n    "plot_confusion_matrix",\n    "plot_roc_curve",\n    "plot_training_history",\n]\n')
print(f'  Ghi: /content/deepfake_detector/utils/__init__.py')

os.makedirs('/content/deepfake_detector/utils', exist_ok=True)
open('/content/deepfake_detector/utils/metrics.py', 'w', encoding='utf-8').write('"""\nEvaluation metrics for deepfake detection.\nIncludes EER, ACER, APCER, NPCER, and other forensic metrics.\n"""\n\nimport numpy as np\nimport math\nfrom typing import Optional, Tuple, List, Dict\nfrom sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score\nimport logging\n\nlogger = logging.getLogger(__name__)\n\n\ndef eval_state(probs: np.ndarray, labels: np.ndarray, thr: float) -> Tuple[int, int, int, int]:\n    """\n    Evaluate predictions at a given threshold.\n\n    Args:\n        probs: Predicted probabilities for the positive class\n        labels: True labels (0 or 1)\n        thr: Decision threshold\n\n    Returns:\n        Tuple of (TN, FN, FP, TP)\n    """\n    predict = probs >= thr\n    TN = np.sum((labels == 0) & (predict == False))\n    FN = np.sum((labels == 1) & (predict == False))\n    FP = np.sum((labels == 0) & (predict == True))\n    TP = np.sum((labels == 1) & (predict == True))\n    return TN, FN, FP, TP\n\n\ndef calculate_metrics(probs: np.ndarray, labels: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:\n    """\n    Calculate comprehensive metrics for deepfake detection.\n\n    Args:\n        probs: Predicted probabilities for real images (1 = real, 0 = fake)\n        labels: True labels (1 = real, 0 = fake)\n        threshold: Decision threshold\n\n    Returns:\n        Dictionary containing APCER, NPCER, ACER, ACC, and other metrics\n    """\n    TN, FN, FP, TP = eval_state(probs, labels, threshold)\n\n    # Attack Presentation Classification Error Rate (false fake as real)\n    APCER = 1.0 if (FP + TN == 0) else FP / float(FP + TN)\n\n    # Normal Presentation Classification Error Rate (false real as fake)\n    # Also known as BPCER (Bona Fide Presentation Classification Error Rate)\n    NPCER = 1.0 if (FN + TP == 0) else FN / float(FN + TP)\n\n    # Average Classification Error Rate\n    ACER = (APCER + NPCER) / 2.0\n\n    # Accuracy\n    ACC = (TP + TN) / (TN + FN + FP + TP) if (TN + FN + FP + TP) > 0 else 0.0\n\n    # Precision and Recall\n    PRECISION = TP / (TP + FP) if (TP + FP) > 0 else 0.0\n    RECALL = TP / (TP + FN) if (TP + FN) > 0 else 0.0\n\n    # F1 Score\n    F1 = 2 * (PRECISION * RECALL) / (PRECISION + RECALL) if (PRECISION + RECALL) > 0 else 0.0\n\n    # Specificity\n    SPECIFICITY = TN / (TN + FP) if (TN + FP) > 0 else 0.0\n\n    metrics = {\n        \'accuracy\': ACC,\n        \'apcer\': APCER,\n        \'npcer\': NPCER,\n        \'acer\': ACER,\n        \'precision\': PRECISION,\n        \'recall\': RECALL,\n        \'f1_score\': F1,\n        \'specificity\': SPECIFICITY,\n        \'tp\': int(TP),\n        \'tn\': int(TN),\n        \'fp\': int(FP),\n        \'fn\': int(FN)\n    }\n\n    return metrics\n\n\ndef get_threshold(probs: np.ndarray, grid_density: int = 10000) -> List[float]:\n    """\n    Generate threshold values for EER calculation.\n\n    Args:\n        probs: Probability array\n        grid_density: Number of threshold points to test\n\n    Returns:\n        List of threshold values\n    """\n    thresholds = [i / float(grid_density) for i in range(grid_density + 1)]\n    thresholds.append(1.1)\n    return thresholds\n\n\ndef get_EER_states(\n    probs: np.ndarray,\n    labels: np.ndarray,\n    grid_density: int = 10000\n) -> Tuple[float, float, List[float], List[float]]:\n    """\n    Calculate Equal Error Rate (EER) and optimal threshold.\n\n    The EER is the point where FAR (False Accept Rate) equals FRR (False Reject Rate).\n\n    Args:\n        probs: Predicted probabilities for positive class\n        labels: True labels\n        grid_density: Number of threshold points to test\n\n    Returns:\n        Tuple of (EER, optimal_threshold, FRR_list, FAR_list)\n    """\n    thresholds = get_threshold(probs, grid_density)\n    min_dist = 1.0\n    min_dist_states = []\n    FRR_list = []\n    FAR_list = []\n\n    for thr in thresholds:\n        TN, FN, FP, TP = eval_state(probs, labels, thr)\n\n        if (FN + TP == 0):\n            FRR = TPR = 1.0\n            FAR = FP / float(FP + TN) if (FP + TN) > 0 else 1.0\n            TNR = TN / float(TN + FP) if (TN + FP) > 0 else 0.0\n        elif (FP + TN == 0):\n            TNR = FAR = 1.0\n            FRR = FN / float(FN + TP)\n            TPR = TP / float(TP + FN)\n        else:\n            FAR = FP / float(FP + TN)\n            FRR = FN / float(FN + TP)\n            TNR = TN / float(TN + FP)\n            TPR = TP / float(TP + FN)\n\n        dist = math.fabs(FRR - FAR)\n        FAR_list.append(FAR)\n        FRR_list.append(FRR)\n\n        if dist <= min_dist:\n            min_dist = dist\n            min_dist_states = [FAR, FRR, thr]\n\n    EER = (min_dist_states[0] + min_dist_states[1]) / 2.0\n    optimal_thr = min_dist_states[2]\n\n    return EER, optimal_thr, FRR_list, FAR_list\n\n\ndef get_HTER_at_thr(probs: np.ndarray, labels: np.ndarray, thr: float) -> float:\n    """\n    Calculate Half Total Error Rate (HTER) at a given threshold.\n\n    HTER is the average of FAR and FRR at a specific threshold.\n\n    Args:\n        probs: Predicted probabilities\n        labels: True labels\n        thr: Decision threshold\n\n    Returns:\n        HTER value\n    """\n    TN, FN, FP, TP = eval_state(probs, labels, thr)\n\n    if (FN + TP == 0):\n        FRR = 1.0\n        FAR = FP / float(FP + TN) if (FP + TN) > 0 else 1.0\n    elif (FP + TN == 0):\n        FAR = 1.0\n        FRR = FN / float(FN + TP)\n    else:\n        FAR = FP / float(FP + TN)\n        FRR = FN / float(FN + TP)\n\n    HTER = (FAR + FRR) / 2.0\n    return HTER\n\n\ndef calculate_comprehensive_metrics(\n    probs: np.ndarray,\n    labels: np.ndarray,\n    preds: Optional[np.ndarray] = None\n) -> Dict[str, float]:\n    """\n    Calculate all metrics including EER, ACER, accuracy, etc.\n\n    Args:\n        probs: Predicted probabilities for positive class\n        labels: True labels\n        preds: Predicted class labels (optional, will be computed from probs if not provided)\n\n    Returns:\n        Dictionary with all metrics\n    """\n    if preds is None:\n        preds = (probs >= 0.5).astype(int)\n\n    # Standard metrics\n    metrics = calculate_metrics(probs, labels, threshold=0.5)\n\n    # EER and optimal threshold\n    EER, optimal_thr, _, _ = get_EER_states(probs, labels)\n    metrics[\'eer\'] = EER\n    metrics[\'optimal_threshold\'] = optimal_thr\n\n    # HTER at threshold 0.5\n    HTER = get_HTER_at_thr(probs, labels, 0.5)\n    metrics[\'hter\'] = HTER\n\n    # Accuracy at optimal threshold\n    optimal_preds = (probs >= optimal_thr).astype(int)\n    optimal_acc = accuracy_score(labels, optimal_preds)\n    metrics[\'accuracy_at_optimal_thr\'] = optimal_acc\n\n    # AUC-ROC if possible\n    try:\n        auc = roc_auc_score(labels, probs)\n        metrics[\'auc_roc\'] = auc\n    except:\n        logger.warning("Could not calculate AUC-ROC")\n        metrics[\'auc_roc\'] = 0.0\n\n    return metrics\n\n\ndef print_metrics(metrics: Dict[str, float], title: str = "Metrics") -> None:\n    """\n    Pretty print metrics.\n\n    Args:\n        metrics: Dictionary of metrics\n        title: Title for the metrics display\n    """\n    print(f"\\n{\'=\'*60}")\n    print(f"{title:^60}")\n    print(f"{\'=\'*60}")\n\n    for key, value in metrics.items():\n        if isinstance(value, (int, np.integer)):\n            print(f"{key:.<30} {value:>10d}")\n        else:\n            print(f"{key:.<30} {value:>10.4f}")\n\n    print(f"{\'=\'*60}\\n")\n')
print(f'  Ghi: /content/deepfake_detector/utils/metrics.py')

os.makedirs('/content/deepfake_detector/utils', exist_ok=True)
open('/content/deepfake_detector/utils/logger.py', 'w', encoding='utf-8').write('"""\nLogging configuration and utilities.\nProvides structured logging for training and evaluation.\n"""\n\nimport logging\nimport sys\nfrom pathlib import Path\nfrom typing import Optional\nfrom datetime import datetime\n\n\ndef setup_logger(\n    name: str = "deepfake_detector",\n    log_file: Optional[str] = None,\n    level: int = logging.INFO,\n    format_string: Optional[str] = None\n) -> logging.Logger:\n    """\n    Set up a logger with console and optional file handlers.\n\n    Args:\n        name: Logger name\n        log_file: Path to log file (optional)\n        level: Logging level\n        format_string: Custom format string (optional)\n\n    Returns:\n        Configured logger instance\n\n    Example:\n        >>> logger = setup_logger("training", log_file="train.log")\n        >>> logger.info("Training started")\n    """\n    logger = logging.getLogger(name)\n    logger.setLevel(level)\n    logger.handlers = []  # Clear existing handlers\n\n    # Default format\n    if format_string is None:\n        format_string = \'[%(asctime)s] [%(name)s] [%(levelname)s] %(message)s\'\n\n    formatter = logging.Formatter(format_string, datefmt=\'%Y-%m-%d %H:%M:%S\')\n\n    # Console handler\n    console_handler = logging.StreamHandler(sys.stdout)\n    console_handler.setLevel(level)\n    console_handler.setFormatter(formatter)\n    logger.addHandler(console_handler)\n\n    # File handler (if specified)\n    if log_file is not None:\n        log_path = Path(log_file)\n        log_path.parent.mkdir(parents=True, exist_ok=True)\n\n        file_handler = logging.FileHandler(log_file, mode=\'a\')\n        file_handler.setLevel(level)\n        file_handler.setFormatter(formatter)\n        logger.addHandler(file_handler)\n\n    return logger\n\n\ndef get_logger(name: str) -> logging.Logger:\n    """\n    Get an existing logger by name.\n\n    Args:\n        name: Logger name\n\n    Returns:\n        Logger instance\n\n    Example:\n        >>> logger = get_logger("deepfake_detector")\n        >>> logger.info("Message")\n    """\n    return logging.getLogger(name)\n\n\nclass TqdmLoggingHandler(logging.Handler):\n    """\n    Custom logging handler that works well with tqdm progress bars.\n\n    Prevents log messages from breaking progress bar display.\n    """\n\n    def __init__(self, level=logging.NOTSET):\n        super().__init__(level)\n\n    def emit(self, record):\n        try:\n            from tqdm import tqdm\n            msg = self.format(record)\n            tqdm.write(msg)\n            self.flush()\n        except Exception:\n            self.handleError(record)\n\n\ndef create_experiment_logger(\n    experiment_name: str,\n    log_dir: str = "logs",\n    level: int = logging.INFO\n) -> logging.Logger:\n    """\n    Create a logger for an experiment with timestamped log file.\n\n    Args:\n        experiment_name: Name of the experiment\n        log_dir: Directory to save logs\n        level: Logging level\n\n    Returns:\n        Configured logger\n\n    Example:\n        >>> logger = create_experiment_logger("efficientnet_b1_training")\n        >>> logger.info("Experiment started")\n    """\n    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")\n    log_filename = f"{experiment_name}_{timestamp}.log"\n    log_path = Path(log_dir) / log_filename\n\n    return setup_logger(\n        name=experiment_name,\n        log_file=str(log_path),\n        level=level\n    )\n')
print(f'  Ghi: /content/deepfake_detector/utils/logger.py')

os.makedirs('/content/deepfake_detector/utils', exist_ok=True)
open('/content/deepfake_detector/utils/visualization.py', 'w', encoding='utf-8').write('"""\nVisualization utilities for model evaluation and training monitoring.\n"""\n\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport numpy as np\nfrom typing import List, Optional, Dict\nfrom pathlib import Path\nimport logging\n\nlogger = logging.getLogger(__name__)\n\n# Set style\nsns.set_style(\'whitegrid\')\nplt.rcParams[\'figure.figsize\'] = (10, 6)\n\n\ndef plot_confusion_matrix(\n    confusion_matrix: np.ndarray,\n    class_names: List[str] = [\'Fake\', \'Real\'],\n    normalize: bool = False,\n    title: str = \'Confusion Matrix\',\n    save_path: Optional[str] = None,\n    show: bool = True\n) -> None:\n    """\n    Plot confusion matrix using seaborn heatmap.\n\n    Args:\n        confusion_matrix: 2x2 confusion matrix\n        class_names: Names of classes\n        normalize: Whether to normalize values\n        title: Plot title\n        save_path: Path to save figure (optional)\n        show: Whether to display the plot\n\n    Example:\n        >>> from sklearn.metrics import confusion_matrix\n        >>> cm = confusion_matrix(y_true, y_pred)\n        >>> plot_confusion_matrix(cm, save_path=\'confusion_matrix.png\')\n    """\n    if normalize:\n        confusion_matrix = confusion_matrix.astype(\'float\') / confusion_matrix.sum(axis=1)[:, np.newaxis]\n        fmt = \'.2%\'\n    else:\n        fmt = \'d\'\n\n    plt.figure(figsize=(8, 6))\n    sns.heatmap(\n        confusion_matrix,\n        annot=True,\n        fmt=fmt,\n        cmap=\'Blues\',\n        xticklabels=[\'Predicted \' + cn for cn in class_names],\n        yticklabels=class_names,\n        linewidths=2,\n        cbar_kws={\'label\': \'Count\' if not normalize else \'Proportion\'}\n    )\n\n    plt.title(title, fontsize=14, fontweight=\'bold\')\n    plt.ylabel(\'True Label\', fontsize=12)\n    plt.xlabel(\'Predicted Label\', fontsize=12)\n    plt.tight_layout()\n\n    if save_path:\n        plt.savefig(save_path, dpi=300, bbox_inches=\'tight\')\n        logger.info(f"Confusion matrix saved to {save_path}")\n\n    if show:\n        plt.show()\n    else:\n        plt.close()\n\n\ndef plot_roc_curve(\n    FRR_list: List[float],\n    FAR_list: List[float],\n    eer: float,\n    title: str = \'ROC Curve (FAR vs FRR)\',\n    save_path: Optional[str] = None,\n    show: bool = True\n) -> None:\n    """\n    Plot ROC curve showing FAR vs FRR.\n\n    Args:\n        FRR_list: False Rejection Rates\n        FAR_list: False Acceptance Rates\n        eer: Equal Error Rate\n        title: Plot title\n        save_path: Path to save figure (optional)\n        show: Whether to display the plot\n\n    Example:\n        >>> EER, thr, FRR_list, FAR_list = get_EER_states(probs, labels)\n        >>> plot_roc_curve(FRR_list, FAR_list, EER, save_path=\'roc_curve.png\')\n    """\n    plt.figure(figsize=(10, 8))\n\n    # Plot ROC curve\n    plt.plot(FAR_list, FRR_list, marker=\'.\', label=f\'ROC Curve (EER={eer:.4f})\', linewidth=2)\n\n    # Plot diagonal (random classifier)\n    plt.plot([0, 1], [0, 1], \'r--\', label=\'Random Classifier\', alpha=0.5)\n\n    # Plot EER point\n    eer_idx = np.argmin(np.abs(np.array(FAR_list) - np.array(FRR_list)))\n    plt.plot(FAR_list[eer_idx], FRR_list[eer_idx], \'ro\', markersize=10, label=f\'EER Point\')\n\n    plt.xlabel(\'False Acceptance Rate (FAR)\', fontsize=12)\n    plt.ylabel(\'False Rejection Rate (FRR)\', fontsize=12)\n    plt.title(title, fontsize=14, fontweight=\'bold\')\n    plt.xlim(0, 1)\n    plt.ylim(0, 1)\n    plt.grid(True, alpha=0.3)\n    plt.legend(loc=\'best\', fontsize=10)\n    plt.tight_layout()\n\n    if save_path:\n        plt.savefig(save_path, dpi=300, bbox_inches=\'tight\')\n        logger.info(f"ROC curve saved to {save_path}")\n\n    if show:\n        plt.show()\n    else:\n        plt.close()\n\n\ndef plot_training_history(\n    history: Dict[str, List[float]],\n    metrics: List[str] = [\'loss\', \'accuracy\'],\n    title: str = \'Training History\',\n    save_path: Optional[str] = None,\n    show: bool = True\n) -> None:\n    """\n    Plot training and validation metrics over epochs.\n\n    Args:\n        history: Dictionary with training history\n                 e.g., {\'train_loss\': [...], \'val_loss\': [...], ...}\n        metrics: List of metrics to plot\n        title: Plot title\n        save_path: Path to save figure (optional)\n        show: Whether to display the plot\n\n    Example:\n        >>> history = {\n        ...     \'train_loss\': [0.5, 0.4, 0.3],\n        ...     \'val_loss\': [0.6, 0.5, 0.4],\n        ...     \'train_accuracy\': [0.7, 0.8, 0.85],\n        ...     \'val_accuracy\': [0.65, 0.75, 0.8]\n        ... }\n        >>> plot_training_history(history, save_path=\'training_history.png\')\n    """\n    num_metrics = len(metrics)\n    fig, axes = plt.subplots(1, num_metrics, figsize=(8 * num_metrics, 6))\n\n    if num_metrics == 1:\n        axes = [axes]\n\n    for idx, metric in enumerate(metrics):\n        ax = axes[idx]\n\n        train_key = f\'train_{metric}\'\n        val_key = f\'val_{metric}\'\n\n        if train_key in history:\n            epochs = range(1, len(history[train_key]) + 1)\n            ax.plot(epochs, history[train_key], \'b-o\', label=f\'Train {metric.capitalize()}\', linewidth=2)\n\n        if val_key in history:\n            epochs = range(1, len(history[val_key]) + 1)\n            ax.plot(epochs, history[val_key], \'r-s\', label=f\'Val {metric.capitalize()}\', linewidth=2)\n\n        ax.set_xlabel(\'Epoch\', fontsize=12)\n        ax.set_ylabel(metric.capitalize(), fontsize=12)\n        ax.set_title(f\'{metric.capitalize()} over Epochs\', fontsize=13, fontweight=\'bold\')\n        ax.legend(loc=\'best\')\n        ax.grid(True, alpha=0.3)\n\n    plt.suptitle(title, fontsize=16, fontweight=\'bold\', y=1.02)\n    plt.tight_layout()\n\n    if save_path:\n        plt.savefig(save_path, dpi=300, bbox_inches=\'tight\')\n        logger.info(f"Training history saved to {save_path}")\n\n    if show:\n        plt.show()\n    else:\n        plt.close()\n\n\ndef plot_sample_predictions(\n    images: np.ndarray,\n    predictions: np.ndarray,\n    labels: np.ndarray,\n    class_names: List[str] = [\'Fake\', \'Real\'],\n    num_samples: int = 16,\n    save_path: Optional[str] = None,\n    show: bool = True\n) -> None:\n    """\n    Plot sample images with predictions and ground truth.\n\n    Args:\n        images: Array of images (N, H, W, C)\n        predictions: Predicted probabilities (N, 2)\n        labels: Ground truth labels (N,)\n        class_names: Names of classes\n        num_samples: Number of samples to display\n        save_path: Path to save figure (optional)\n        show: Whether to display the plot\n\n    Example:\n        >>> plot_sample_predictions(\n        ...     images, predictions, labels,\n        ...     num_samples=9,\n        ...     save_path=\'predictions.png\'\n        ... )\n    """\n    num_samples = min(num_samples, len(images))\n    grid_size = int(np.ceil(np.sqrt(num_samples)))\n\n    fig, axes = plt.subplots(grid_size, grid_size, figsize=(15, 15))\n    axes = axes.flatten()\n\n    for idx in range(num_samples):\n        ax = axes[idx]\n\n        # Display image\n        img = images[idx]\n        if img.shape[0] == 3:  # If channels first\n            img = np.transpose(img, (1, 2, 0))\n\n        # Denormalize if needed\n        if img.max() <= 1.0:\n            img = (img - img.min()) / (img.max() - img.min())\n\n        ax.imshow(img)\n\n        # Get prediction and label\n        pred_class = np.argmax(predictions[idx])\n        pred_prob = predictions[idx][pred_class]\n        true_class = labels[idx]\n\n        # Set title with color based on correctness\n        color = \'green\' if pred_class == true_class else \'red\'\n        title = f\'Pred: {class_names[pred_class]} ({pred_prob:.2f})\\nTrue: {class_names[true_class]}\'\n        ax.set_title(title, color=color, fontsize=10, fontweight=\'bold\')\n        ax.axis(\'off\')\n\n    # Hide extra subplots\n    for idx in range(num_samples, len(axes)):\n        axes[idx].axis(\'off\')\n\n    plt.suptitle(\'Sample Predictions\', fontsize=16, fontweight=\'bold\')\n    plt.tight_layout()\n\n    if save_path:\n        plt.savefig(save_path, dpi=300, bbox_inches=\'tight\')\n        logger.info(f"Sample predictions saved to {save_path}")\n\n    if show:\n        plt.show()\n    else:\n        plt.close()\n')
print(f'  Ghi: /content/deepfake_detector/utils/visualization.py')

os.makedirs('/content/deepfake_detector/config', exist_ok=True)
open('/content/deepfake_detector/config/__init__.py', 'w', encoding='utf-8').write('"""Configuration management module."""\n\nfrom deepfake_detector.config.config import Config, load_config, save_config\n\n__all__ = ["Config", "load_config", "save_config"]\n')
print(f'  Ghi: /content/deepfake_detector/config/__init__.py')

os.makedirs('/content/deepfake_detector/config', exist_ok=True)
open('/content/deepfake_detector/config/config.py', 'w', encoding='utf-8').write('"""\nConfiguration management for training and inference.\nSupports YAML and JSON configuration files.\n"""\n\nimport json\nimport yaml\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional, Union\nfrom dataclasses import dataclass, asdict, field\nimport logging\n\nlogger = logging.getLogger(__name__)\n\n\n@dataclass\nclass Config:\n    """\n    Configuration class for deepfake detection pipeline.\n\n    All parameters are documented and have sensible defaults.\n    """\n    # Model Configuration\n    model_name: str = \'efficientnet-b1\'\n    num_classes: int = 2\n    dropout_rate: float = 0.5\n    pretrained: bool = True\n\n    # Training Configuration\n    batch_size: int = 32\n    num_epochs: int = 20\n    learning_rate: float = 8e-4\n    weight_decay: float = 1e-3\n    warmup_epochs: int = 5\n\n    # Data Configuration\n    image_size: int = 240\n    num_workers: int = 4\n    pin_memory: bool = True\n    use_heavy_augmentation: bool = False\n\n    # Paths\n    train_real_dirs: list = field(default_factory=list)\n    train_fake_dirs: list = field(default_factory=list)\n    val_real_dirs: list = field(default_factory=list)\n    val_fake_dirs: list = field(default_factory=list)\n    test_real_dirs: list = field(default_factory=list)\n    test_fake_dirs: list = field(default_factory=list)\n\n    # Checkpoint Configuration\n    checkpoint_dir: str = \'checkpoints\'\n    save_every_n_epochs: int = 1\n    keep_last_n_checkpoints: int = 5\n\n    # Logging Configuration\n    log_dir: str = \'logs\'\n    results_dir: str = \'results\'\n    experiment_name: str = \'deepfake_detection\'\n\n    # Device Configuration\n    device: str = \'cuda\'\n    mixed_precision: bool = True\n\n    # Evaluation Configuration\n    test_batch_size: int = 100\n    eer_grid_density: int = 10000\n\n    # Resume Training\n    resume_from_checkpoint: Optional[str] = None\n    start_epoch: int = 0\n\n    def __post_init__(self):\n        """Validate configuration after initialization."""\n        if self.batch_size <= 0:\n            raise ValueError(f"batch_size must be positive, got {self.batch_size}")\n        if self.num_epochs <= 0:\n            raise ValueError(f"num_epochs must be positive, got {self.num_epochs}")\n        if not 0 < self.learning_rate < 1:\n            raise ValueError(f"learning_rate must be in (0, 1), got {self.learning_rate}")\n\n    def to_dict(self) -> Dict[str, Any]:\n        """Convert config to dictionary."""\n        return asdict(self)\n\n    def save(self, filepath: Union[str, Path]) -> None:\n        """\n        Save configuration to file.\n\n        Args:\n            filepath: Path to save configuration (supports .json and .yaml)\n\n        Example:\n            >>> config = Config()\n            >>> config.save(\'config.yaml\')\n        """\n        filepath = Path(filepath)\n        filepath.parent.mkdir(parents=True, exist_ok=True)\n\n        config_dict = self.to_dict()\n\n        if filepath.suffix in [\'.yaml\', \'.yml\']:\n            with open(filepath, \'w\') as f:\n                yaml.dump(config_dict, f, default_flow_style=False, indent=2)\n        elif filepath.suffix == \'.json\':\n            with open(filepath, \'w\') as f:\n                json.dump(config_dict, f, indent=2)\n        else:\n            raise ValueError(f"Unsupported file format: {filepath.suffix}")\n\n        logger.info(f"Configuration saved to {filepath}")\n\n    @classmethod\n    def from_dict(cls, config_dict: Dict[str, Any]) -> \'Config\':\n        """\n        Create Config from dictionary.\n\n        Args:\n            config_dict: Dictionary with configuration parameters\n\n        Returns:\n            Config instance\n\n        Example:\n            >>> config_dict = {\'model_name\': \'efficientnet-b0\', \'batch_size\': 64}\n            >>> config = Config.from_dict(config_dict)\n        """\n        return cls(**config_dict)\n\n    @classmethod\n    def load(cls, filepath: Union[str, Path]) -> \'Config\':\n        """\n        Load configuration from file.\n\n        Args:\n            filepath: Path to configuration file\n\n        Returns:\n            Config instance\n\n        Example:\n            >>> config = Config.load(\'config.yaml\')\n        """\n        filepath = Path(filepath)\n\n        if not filepath.exists():\n            raise FileNotFoundError(f"Configuration file not found: {filepath}")\n\n        if filepath.suffix in [\'.yaml\', \'.yml\']:\n            with open(filepath, \'r\') as f:\n                config_dict = yaml.safe_load(f)\n        elif filepath.suffix == \'.json\':\n            with open(filepath, \'r\') as f:\n                config_dict = json.load(f)\n        else:\n            raise ValueError(f"Unsupported file format: {filepath.suffix}")\n\n        logger.info(f"Configuration loaded from {filepath}")\n        return cls.from_dict(config_dict)\n\n    def __repr__(self) -> str:\n        """String representation of configuration."""\n        lines = ["Configuration:"]\n        for key, value in self.to_dict().items():\n            lines.append(f"  {key}: {value}")\n        return "\\n".join(lines)\n\n\ndef load_config(filepath: Union[str, Path]) -> Config:\n    """\n    Load configuration from file.\n\n    Args:\n        filepath: Path to configuration file\n\n    Returns:\n        Config instance\n\n    Example:\n        >>> config = load_config(\'config.yaml\')\n    """\n    return Config.load(filepath)\n\n\ndef save_config(config: Config, filepath: Union[str, Path]) -> None:\n    """\n    Save configuration to file.\n\n    Args:\n        config: Configuration instance\n        filepath: Path to save configuration\n\n    Example:\n        >>> config = Config()\n        >>> save_config(config, \'config.yaml\')\n    """\n    config.save(filepath)\n\n\ndef create_default_config(filepath: Union[str, Path]) -> Config:\n    """\n    Create and save a default configuration file.\n\n    Args:\n        filepath: Path to save default configuration\n\n    Returns:\n        Default Config instance\n\n    Example:\n        >>> config = create_default_config(\'default_config.yaml\')\n    """\n    config = Config()\n    config.save(filepath)\n    logger.info(f"Default configuration created at {filepath}")\n    return config\n')
print(f'  Ghi: /content/deepfake_detector/config/config.py')

os.makedirs('/content/scripts', exist_ok=True)
open('/content/scripts/extract_faces.py', 'w', encoding='utf-8').write('#!/usr/bin/env python3\n"""\nFace Extraction Script using MTCNN\nExtracts faces from videos and images for deepfake detection.\n"""\n\nimport argparse\nimport os\nimport glob\nimport time\nimport cv2\nimport torch\nimport numpy as np\nfrom pathlib import Path\nfrom typing import List, Tuple\nfrom tqdm import tqdm\nimport logging\n\ntry:\n    from facenet_pytorch import MTCNN\nexcept ImportError:\n    print("Error: facenet-pytorch not installed. Install with: pip install facenet-pytorch")\n    exit(1)\n\n# Setup logging\nlogging.basicConfig(\n    level=logging.INFO,\n    format=\'[%(asctime)s] [%(levelname)s] %(message)s\',\n    datefmt=\'%Y-%m-%d %H:%M:%S\'\n)\nlogger = logging.getLogger(__name__)\n\n\nclass FastMTCNN:\n    """\n    Fast MTCNN implementation for efficient face detection.\n\n    Processes frames with stride to balance speed and accuracy.\n    """\n\n    def __init__(\n        self,\n        stride: int = 1,\n        resize: float = 1.0,\n        margin: int = 50,\n        min_face_size: int = 100,\n        thresholds: List[float] = [0.6, 0.7, 0.7],\n        factor: float = 0.7,\n        post_process: bool = True,\n        select_largest: bool = True,\n        keep_all: bool = True,\n        device: str = \'cuda\',\n        square_crop: bool = True,\n        square_margin: float = 0.30,\n        pad_mode: str = \'reflect\',\n    ):\n        """\n        Initialize FastMTCNN.\n\n        Args:\n            stride: Detection stride (detect every N frames)\n            resize: Frame resize factor\n            margin: Margin around detected face\n            min_face_size: Minimum face size to detect\n            thresholds: MTCNN detection thresholds\n            factor: MTCNN scale factor\n            post_process: Whether to post-process detections\n            select_largest: Select only largest face\n            keep_all: Keep all detected faces\n            device: Device to run on (\'cuda\' or \'cpu\')\n        """\n        self.stride = stride\n        self.resize = resize\n        self.square_crop = square_crop\n        self.square_margin = square_margin\n        self.pad_mode = pad_mode\n\n        self.mtcnn = MTCNN(\n            margin=margin,\n            min_face_size=min_face_size,\n            thresholds=thresholds,\n            factor=factor,\n            post_process=post_process,\n            select_largest=select_largest,\n            keep_all=keep_all,\n            device=device\n        )\n\n        logger.info(f"FastMTCNN initialized on {device}")\n\n    def __call__(self, frames: List, output_dir: str, prefix: str = "face") -> int:\n        """\n        Extract faces from frames.\n\n        Args:\n            frames: List of frames (numpy arrays)\n            output_dir: Output directory for extracted faces\n            prefix: Prefix for saved images\n\n        Returns:\n            Number of faces extracted\n        """\n        if self.resize != 1.0:\n            frames = [\n                cv2.resize(f, (int(f.shape[1] * self.resize), int(f.shape[0] * self.resize)))\n                for f in frames\n            ]\n\n        # Detect faces\n        boxes, probs = self.mtcnn.detect(frames[::self.stride])\n\n        faces_count = 0\n        for i, frame in enumerate(frames):\n            box_ind = int(i / self.stride)\n\n            if boxes[box_ind] is None:\n                continue\n\n            for box in boxes[box_ind]:\n                box = [int(b) for b in box]\n\n                # Extract face region (optionally force square crop with padding).\n                x1, y1, x2, y2 = box\n                if self.square_crop:\n                    w = x2 - x1\n                    h = y2 - y1\n                    side = max(w, h)\n                    side = int(side * (1.0 + self.square_margin))\n                    cx = (x1 + x2) / 2.0\n                    cy = (y1 + y2) / 2.0\n                    sx1 = int(round(cx - side / 2.0))\n                    sy1 = int(round(cy - side / 2.0))\n                    sx2 = sx1 + side\n                    sy2 = sy1 + side\n\n                    pad_left = max(0, -sx1)\n                    pad_top = max(0, -sy1)\n                    pad_right = max(0, sx2 - frame.shape[1])\n                    pad_bottom = max(0, sy2 - frame.shape[0])\n\n                    if pad_left or pad_top or pad_right or pad_bottom:\n                        frame_pad = cv2.copyMakeBorder(\n                            frame,\n                            pad_top,\n                            pad_bottom,\n                            pad_left,\n                            pad_right,\n                            borderType=cv2.BORDER_REFLECT if self.pad_mode == \'reflect\' else cv2.BORDER_CONSTANT,\n                        )\n                    else:\n                        frame_pad = frame\n\n                    sx1p = sx1 + pad_left\n                    sy1p = sy1 + pad_top\n                    sx2p = sx2 + pad_left\n                    sy2p = sy2 + pad_top\n                    face = frame_pad[sy1p:sy2p, sx1p:sx2p]\n                else:\n                    face = frame[y1:y2, x1:x2]\n\n                # Validate face\n                if len(face) == 0 or face.shape[0] < 10 or face.shape[1] < 10:\n                    continue\n\n                # Save face\n                timestamp = time.time()\n                filename = f"{prefix}-{timestamp:.6f}.jpg"\n                filepath = os.path.join(output_dir, filename)\n\n                # Convert RGB to BGR for cv2\n                face_bgr = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)\n                cv2.imwrite(filepath, face_bgr)\n\n                faces_count += 1\n\n        return faces_count\n\n\ndef process_video(\n    video_path: str,\n    output_dir: str,\n    mtcnn: FastMTCNN,\n    batch_size: int = 60,\n    frame_skip: int = 30,\n    frames_per_video: int = 0,\n    sampling_strategy: str = "stride",\n) -> Tuple[int, int]:\n    """\n    Process a single video file.\n\n    Args:\n        video_path: Path to video file\n        output_dir: Output directory\n        mtcnn: FastMTCNN instance\n        batch_size: Number of frames to process at once\n        frame_skip: Process every Nth frame (stride mode)\n        frames_per_video: If > 0, override sampling by taking N frames\n                          uniformly across the whole video.\n        sampling_strategy: \'stride\' or \'uniform\'\n\n    Returns:\n        Tuple of (frames_processed, faces_detected)\n    """\n    cap = cv2.VideoCapture(video_path)\n    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n\n    frames = []\n    frames_processed = 0\n    faces_detected = 0\n\n    # Precompute targets for uniform sampling (exact N frames).\n    uniform_indices: List[int] = []\n    if sampling_strategy == "uniform" and frames_per_video > 0 and total_frames > 0:\n        if total_frames <= frames_per_video:\n            uniform_indices = list(range(total_frames))\n        else:\n            uniform_indices = np.linspace(0, total_frames - 1, frames_per_video).astype(int).tolist()\n    uniform_ptr = 0\n    uniform_target = uniform_indices[uniform_ptr] if uniform_indices else None\n\n    for frame_idx in range(total_frames):\n        ret, frame = cap.read()\n\n        if not ret:\n            break\n\n        take = False\n        if uniform_target is not None:\n            if frame_idx == uniform_target:\n                take = True\n                uniform_ptr += 1\n                uniform_target = uniform_indices[uniform_ptr] if uniform_ptr < len(uniform_indices) else None\n        else:\n            # Stride mode: take every Nth frame and always the last frame.\n            if frame_idx % frame_skip == 0 or frame_idx == total_frames - 1:\n                take = True\n\n        if take:\n            # Convert BGR to RGB\n            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n            frames.append(frame_rgb)\n\n            if len(frames) >= batch_size or frame_idx == total_frames - 1:\n                # Process batch\n                video_name = Path(video_path).stem\n                faces = mtcnn(frames, output_dir, prefix=video_name)\n\n                frames_processed += len(frames)\n                faces_detected += faces\n\n                frames = []\n\n    cap.release()\n    return frames_processed, faces_detected\n\n\ndef process_image(\n    image_path: str,\n    output_dir: str,\n    mtcnn: FastMTCNN\n) -> Tuple[int, int]:\n    """\n    Process a single image file.\n\n    Args:\n        image_path: Path to image file\n        output_dir: Output directory\n        mtcnn: FastMTCNN instance\n\n    Returns:\n        Tuple of (images_processed, faces_detected)\n    """\n    image = cv2.imread(image_path)\n\n    if image is None:\n        logger.warning(f"Failed to read image: {image_path}")\n        return 0, 0\n\n    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)\n\n    image_name = Path(image_path).stem\n    faces = mtcnn([image_rgb], output_dir, prefix=image_name)\n\n    return 1, faces\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description=\'Extract faces from videos and images using MTCNN\',\n        formatter_class=argparse.ArgumentDefaultsHelpFormatter\n    )\n\n    # Input/Output\n    parser.add_argument(\'--input-dir\', type=str, required=True,\n                        help=\'Input directory containing videos/images\')\n    parser.add_argument(\'--output-dir\', type=str, required=True,\n                        help=\'Output directory for extracted faces\')\n\n    # Processing options\n    parser.add_argument(\'--mode\', type=str, choices=[\'video\', \'image\'], default=\'video\',\n                        help=\'Processing mode\')\n    parser.add_argument(\'--batch-size\', type=int, default=60,\n                        help=\'Batch size for processing\')\n    parser.add_argument(\'--frame-skip\', type=int, default=30,\n                        help=\'Process every Nth frame (video only)\')\n    parser.add_argument(\'--sampling-strategy\', type=str, default=\'stride\', choices=[\'stride\', \'uniform\'],\n                        help=\'Video frame sampling strategy\')\n    parser.add_argument(\'--frames-per-video\', type=int, default=0,\n                        help=\'If > 0 and sampling-strategy=uniform, take exactly N frames per video\')\n\n    # MTCNN parameters\n    parser.add_argument(\'--stride\', type=int, default=1,\n                        help=\'Detection stride\')\n    parser.add_argument(\'--margin\', type=int, default=50,\n                        help=\'Margin around detected face\')\n    parser.add_argument(\'--min-face-size\', type=int, default=100,\n                        help=\'Minimum face size to detect\')\n\n    # Crop options\n    parser.add_argument(\'--square-crop\', action=\'store_true\', default=True,\n                        help=\'Force square crop around detected face (recommended)\')\n    parser.add_argument(\'--no-square-crop\', action=\'store_false\', dest=\'square_crop\',\n                        help=\'Disable square crop around detected face\')\n    parser.add_argument(\'--square-margin\', type=float, default=0.30,\n                        help=\'Extra padding fraction for square crop\')\n    parser.add_argument(\'--pad-mode\', type=str, default=\'reflect\',\n                        choices=[\'reflect\', \'constant\'],\n                        help=\'Padding mode for out-of-bound crops\')\n\n    # Device\n    parser.add_argument(\'--device\', type=str, default=\'cuda\',\n                        choices=[\'cuda\', \'cpu\'],\n                        help=\'Device to use\')\n\n    args = parser.parse_args()\n\n    # Create output directory\n    os.makedirs(args.output_dir, exist_ok=True)\n\n    # Check device availability\n    device = args.device\n    if device == \'cuda\' and not torch.cuda.is_available():\n        logger.warning("CUDA not available, using CPU")\n        device = \'cpu\'\n\n    # Initialize MTCNN\n    logger.info("Initializing MTCNN...")\n    mtcnn = FastMTCNN(\n        stride=args.stride,\n        margin=args.margin,\n        min_face_size=args.min_face_size,\n        device=device,\n        square_crop=args.square_crop,\n        square_margin=args.square_margin,\n        pad_mode=args.pad_mode,\n    )\n\n    # Get input files\n    if args.mode == \'video\':\n        patterns = [\'*.mp4\', \'*.avi\', \'*.mov\', \'*.mkv\']\n    else:\n        patterns = [\'*.jpg\', \'*.jpeg\', \'*.png\']\n\n    input_files = []\n    for pattern in patterns:\n        input_files.extend(glob.glob(os.path.join(args.input_dir, pattern)))\n\n    if not input_files:\n        logger.error(f"No files found in {args.input_dir}")\n        return\n\n    logger.info(f"Found {len(input_files)} files to process")\n\n    # Process files\n    total_frames = 0\n    total_faces = 0\n\n    for filepath in tqdm(input_files, desc=\'Processing files\'):\n        try:\n            if args.mode == \'video\':\n                frames, faces = process_video(\n                    filepath,\n                    args.output_dir,\n                    mtcnn,\n                    args.batch_size,\n                    args.frame_skip,\n                    frames_per_video=args.frames_per_video,\n                    sampling_strategy=args.sampling_strategy,\n                )\n            else:\n                frames, faces = process_image(\n                    filepath,\n                    args.output_dir,\n                    mtcnn\n                )\n\n            total_frames += frames\n            total_faces += faces\n\n        except Exception as e:\n            logger.error(f"Error processing {filepath}: {e}")\n            continue\n\n    logger.info(f"Processing complete!")\n    logger.info(f"Total frames processed: {total_frames}")\n    logger.info(f"Total faces detected: {total_faces}")\n    logger.info(f"Faces saved to: {args.output_dir}")\n\n\nif __name__ == \'__main__\':\n    main()\n')
print(f'  Ghi: /content/scripts/extract_faces.py')

os.makedirs('/content/scripts', exist_ok=True)
open('/content/scripts/extract_from_manifest.py', 'w', encoding='utf-8').write('#!/usr/bin/env python3\n"""\nExtract face crops from videos according to a split manifest JSON.\n\nThis script helps enforce per-video frame quotas for balanced REAL/FAKE-any:\n - e.g. original -> 24 frames/video\n - fake classes -> 4 frames/video\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Dict, List\n\nimport torch\nfrom tqdm import tqdm\n\nfrom extract_faces import FastMTCNN, process_video\n\n\ndef _load_manifest(path: Path) -> Dict:\n    with path.open("r", encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef _resolve_quota(class_name: str, real_class: str, real_quota: int, fake_quota: int) -> int:\n    return real_quota if class_name == real_class else fake_quota\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Extract face crops using split manifest",\n        formatter_class=argparse.ArgumentDefaultsHelpFormatter,\n    )\n    parser.add_argument("--manifest", type=str, required=True, help="Path to split manifest JSON")\n    parser.add_argument("--split", type=str, choices=["train", "val", "test"], required=True, help="Split to extract")\n    parser.add_argument("--output-root", type=str, required=True, help="Root output folder for extracted faces")\n    parser.add_argument("--real-class", type=str, default="original", help="Name of the real class")\n    parser.add_argument("--real-frames-per-video", type=int, default=24, help="Frames per real video")\n    parser.add_argument("--fake-frames-per-video", type=int, default=4, help="Frames per fake video")\n    parser.add_argument("--batch-size", type=int, default=60, help="Batch size passed to face extractor")\n    parser.add_argument("--sampling-strategy", type=str, default="uniform", choices=["uniform", "stride"], help="Sampling strategy")\n    parser.add_argument("--frame-skip", type=int, default=30, help="Stride mode: process every Nth frame")\n    parser.add_argument("--stride", type=int, default=1, help="MTCNN detect stride")\n    parser.add_argument("--margin", type=int, default=50, help="MTCNN margin")\n    parser.add_argument("--min-face-size", type=int, default=100, help="MTCNN minimum face size")\n    parser.add_argument("--square-margin", type=float, default=0.30, help="Square crop margin fraction")\n    parser.add_argument("--pad-mode", type=str, default="reflect", choices=["reflect", "constant"], help="Padding mode for square crop")\n    parser.add_argument("--device", type=str, default="cuda", choices=["cuda", "cpu"], help="Device")\n    args = parser.parse_args()\n\n    manifest_path = Path(args.manifest).resolve()\n    if not manifest_path.exists():\n        raise FileNotFoundError(f"Manifest not found: {manifest_path}")\n\n    if args.device == "cuda" and not torch.cuda.is_available():\n        print("CUDA not available, switching to CPU")\n        args.device = "cpu"\n\n    manifest = _load_manifest(manifest_path)\n    split_data: Dict[str, List[str]] = manifest["data"][args.split]\n\n    output_root = Path(args.output_root).resolve()\n    output_root.mkdir(parents=True, exist_ok=True)\n\n    mtcnn = FastMTCNN(\n        stride=args.stride,\n        margin=args.margin,\n        min_face_size=args.min_face_size,\n        device=args.device,\n        square_crop=True,\n        square_margin=args.square_margin,\n        pad_mode=args.pad_mode,\n    )\n\n    grand_total_frames = 0\n    grand_total_faces = 0\n\n    classes = sorted(split_data.keys())\n    for class_name in classes:\n        video_paths = split_data[class_name]\n        class_out = output_root / args.split / class_name\n        class_out.mkdir(parents=True, exist_ok=True)\n\n        frames_per_video = _resolve_quota(\n            class_name,\n            real_class=args.real_class,\n            real_quota=args.real_frames_per_video,\n            fake_quota=args.fake_frames_per_video,\n        )\n        print(\n            f"\\n[{args.split}] class={class_name} videos={len(video_paths)} "\n            f"frames_per_video={frames_per_video}"\n        )\n\n        for video_path in tqdm(video_paths, desc=f"{args.split}:{class_name}"):\n            try:\n                frames, faces = process_video(\n                    video_path=video_path,\n                    output_dir=str(class_out),\n                    mtcnn=mtcnn,\n                    batch_size=args.batch_size,\n                    frame_skip=args.frame_skip,\n                    frames_per_video=frames_per_video,\n                    sampling_strategy=args.sampling_strategy,\n                )\n                grand_total_frames += frames\n                grand_total_faces += faces\n            except Exception as exc:\n                print(f"Error processing {video_path}: {exc}")\n\n    print("\\nExtraction complete.")\n    print(f"Total sampled frames: {grand_total_frames}")\n    print(f"Total extracted faces: {grand_total_faces}")\n    print(f"Saved to: {output_root}")\n\n\nif __name__ == "__main__":\n    main()\n\n')
print(f'  Ghi: /content/scripts/extract_from_manifest.py')

os.makedirs('/content/scripts', exist_ok=True)
open('/content/scripts/create_split_manifest_from_csv.py', 'w', encoding='utf-8').write('#!/usr/bin/env python3\n"""\nCreate reproducible train/val/test split manifest from CSV metadata files.\n\nCSV expected columns (case-insensitive, leading/trailing spaces allowed):\n - File Path\n - Label  (optional, used for sanity checks only)\n\nExample row:\n  DeepFakeDetection/01_02__meeting_serious__YVGY8LOK.mp4, FAKE, ...\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport random\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple\n\nimport pandas as pd\n\n\ndef _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:\n    df = df.copy()\n    df.columns = [str(c).strip() for c in df.columns]\n    return df\n\n\ndef _find_col(df: pd.DataFrame, candidates: List[str]) -> str:\n    lower_map = {c.lower(): c for c in df.columns}\n    for cand in candidates:\n        key = cand.lower()\n        if key in lower_map:\n            return lower_map[key]\n    raise ValueError(f"None of columns {candidates} found in CSV columns: {list(df.columns)}")\n\n\ndef _read_csv_flexible(csv_path: Path) -> pd.DataFrame:\n    # Try with Python\'s auto separator inference first.\n    try:\n        df = pd.read_csv(csv_path, sep=None, engine="python")\n        return _normalize_columns(df)\n    except Exception:\n        pass\n\n    # Fallbacks for common separators.\n    for sep in [",", "\\t", ";"]:\n        try:\n            df = pd.read_csv(csv_path, sep=sep)\n            return _normalize_columns(df)\n        except Exception:\n            continue\n    raise ValueError(f"Could not parse CSV file: {csv_path}")\n\n\ndef _split_items(\n    items: List[str],\n    train_ratio: float,\n    val_ratio: float,\n    test_ratio: float,\n    rng: random.Random,\n) -> Tuple[List[str], List[str], List[str]]:\n    if not items:\n        return [], [], []\n    temp = items[:]\n    rng.shuffle(temp)\n    n = len(temp)\n    n_train = int(n * train_ratio)\n    n_val = int(n * val_ratio)\n    n_test = n - n_train - n_val\n    train = temp[:n_train]\n    val = temp[n_train:n_train + n_val]\n    test = temp[n_train + n_val:n_train + n_val + n_test]\n    return train, val, test\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Create split manifest directly from CSV metadata",\n        formatter_class=argparse.ArgumentDefaultsHelpFormatter,\n    )\n    parser.add_argument("--dataset-root", type=str, required=True, help="Dataset root folder containing video subfolders")\n    parser.add_argument("--csv-dir", type=str, required=True, help="Folder containing CSV files")\n    parser.add_argument("--csv-files", type=str, nargs="*", default=None, help="Optional explicit CSV filenames")\n    parser.add_argument("--train-ratio", type=float, default=0.72,\n                        help="Train split ratio (default 0.72 → ~720/1000 videos)")\n    parser.add_argument("--val-ratio", type=float, default=0.14,\n                        help="Val split ratio  (default 0.14 → ~140/1000 videos)")\n    parser.add_argument("--test-ratio", type=float, default=0.14,\n                        help="Test split ratio (default 0.14 → ~140/1000 videos)")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--output", type=str, required=True, help="Output split manifest JSON")\n    args = parser.parse_args()\n\n    ratio_sum = args.train_ratio + args.val_ratio + args.test_ratio\n    if abs(ratio_sum - 1.0) > 1e-6:\n        raise ValueError(f"Split ratios must sum to 1.0, got {ratio_sum}")\n\n    dataset_root = Path(args.dataset_root).resolve()\n    csv_dir = Path(args.csv_dir).resolve()\n    if not dataset_root.exists():\n        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")\n    if not csv_dir.exists():\n        raise FileNotFoundError(f"CSV dir not found: {csv_dir}")\n\n    if args.csv_files:\n        csv_paths = [(csv_dir / f).resolve() for f in args.csv_files]\n    else:\n        csv_paths = sorted(csv_dir.glob("*.csv"))\n\n    if not csv_paths:\n        raise FileNotFoundError(f"No CSV files found in: {csv_dir}")\n\n    rng = random.Random(args.seed)\n\n    # class_name -> list[absolute_video_path]\n    by_class: Dict[str, List[str]] = {}\n\n    for csv_path in csv_paths:\n        df = _read_csv_flexible(csv_path)\n        try:\n            file_col = _find_col(df, ["File Path", "filepath", "path", "file_path"])\n        except ValueError:\n            print(f"[SKIP] {csv_path.name} — khong co cot \'File Path\', bo qua (co the la file thong ke)")\n            continue\n        label_col = None\n        try:\n            label_col = _find_col(df, ["Label", "label"])\n        except Exception:\n            label_col = None\n\n        for _, row in df.iterrows():\n            rel_path = str(row[file_col]).strip()\n            if not rel_path:\n                continue\n            # Expected path shape: ClassName/filename.mp4\n            rel = Path(rel_path)\n            if len(rel.parts) < 2:\n                continue\n            class_name = rel.parts[0]\n            abs_path = (dataset_root / rel).resolve()\n            by_class.setdefault(class_name, []).append(str(abs_path))\n\n            # Optional sanity check: warn if class seems inconsistent with label.\n            if label_col is not None:\n                label = str(row[label_col]).strip().upper()\n                if class_name.lower() == "original" and label == "FAKE":\n                    print(f"[WARN] original class with FAKE label in {csv_path.name}: {rel_path}")\n\n    # Deduplicate paths per class while preserving order.\n    for class_name in list(by_class.keys()):\n        seen = set()\n        uniq = []\n        for p in by_class[class_name]:\n            if p not in seen:\n                uniq.append(p)\n                seen.add(p)\n        by_class[class_name] = uniq\n\n    classes = sorted(by_class.keys())\n    if not classes:\n        raise ValueError("No video entries found from CSV files.")\n\n    manifest = {\n        "seed": args.seed,\n        "splits": {\n            "train": args.train_ratio,\n            "val": args.val_ratio,\n            "test": args.test_ratio,\n        },\n        "classes": classes,\n        "data": {\n            "train": {},\n            "val": {},\n            "test": {},\n        },\n    }\n\n    for class_name in classes:\n        items = by_class[class_name]\n        train, val, test = _split_items(\n            items,\n            args.train_ratio,\n            args.val_ratio,\n            args.test_ratio,\n            rng,\n        )\n        manifest["data"]["train"][class_name] = train\n        manifest["data"]["val"][class_name] = val\n        manifest["data"]["test"][class_name] = test\n        print(\n            f"{class_name}: total={len(items)} train={len(train)} val={len(val)} test={len(test)}"\n        )\n\n    output = Path(args.output).resolve()\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", encoding="utf-8") as f:\n        json.dump(manifest, f, indent=2)\n    print(f"Saved split manifest to: {output}")\n\n\nif __name__ == "__main__":\n    main()\n\n')
print(f'  Ghi: /content/scripts/create_split_manifest_from_csv.py')

os.makedirs('/content/scripts', exist_ok=True)
open('/content/scripts/train.py', 'w', encoding='utf-8').write('#!/usr/bin/env python3\n"""\nTraining script for DeepFake Detection\nRobust training with checkpointing, logging, and monitoring.\n"""\n\nimport argparse\nimport os\nimport sys\nfrom pathlib import Path\n\n# Add parent directory to path\nsys.path.insert(0, str(Path(__file__).parent.parent))\n\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\nimport numpy as np\nfrom tqdm import tqdm\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score\nfrom torch.utils.data import WeightedRandomSampler\nfrom scipy.special import softmax\n\n# Import from our package\nfrom deepfake_detector.models import DeepFakeDetector, TriStreamDeepFakeDetector\nfrom deepfake_detector.data import create_combined_dataset, get_train_transforms, get_val_transforms, create_dataloaders\nfrom deepfake_detector.utils import setup_logger, calculate_comprehensive_metrics, plot_confusion_matrix, plot_training_history\nfrom deepfake_detector.config import Config, load_config\n\nfrom transformers import AdamW, get_cosine_schedule_with_warmup\nimport torch.nn.functional as F\n\n\n# ── Focal Loss ────────────────────────────────────────────────────────────────\n\nclass FocalLoss(nn.Module):\n    """\n    Focal Loss cho multi-class classification (dùng với CrossEntropy).\n\n    FL(pt) = -alpha * (1 - pt)^gamma * log(pt)\n\n    - gamma > 0: giảm trọng số của easy examples, tập trung vào hard examples.\n    - Hiệu quả với imbalanced dataset khi kết hợp cùng WeightedRandomSampler.\n    - gamma=2.0 là giá trị tốt nhất theo paper gốc (Lin et al., 2017).\n    """\n\n    def __init__(self, gamma: float = 2.0, weight=None):\n        super().__init__()\n        self.gamma = gamma\n        self.weight = weight  # per-class weight tensor (optional)\n\n    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\n        # logits: [B, C],  targets: [B]\n        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")  # [B]\n        pt = torch.exp(-ce)           # probability of correct class  ∈ (0, 1]\n        focal = (1.0 - pt) ** self.gamma * ce\n        return focal.mean()\n\n\ndef _effnet_input_size(model_name: str) -> int:\n    sizes = {"b0": 224, "b1": 240, "b2": 260, "b3": 300,\n             "b4": 380, "b5": 456, "b6": 528, "b7": 600}\n    name = model_name.lower()\n    for k, v in sizes.items():\n        if k in name:\n            return v\n    return 224\n\n\ndef _effnet_feat_dim(model_name: str) -> int:\n    dims = {"b0": 1280, "b1": 1280, "b2": 1408, "b3": 1536,\n            "b4": 1792, "b5": 2048, "b6": 2304, "b7": 2560}\n    name = model_name.lower()\n    for k, v in dims.items():\n        if k in name:\n            return v\n    return 1280\n\n\ndef train_epoch(model, dataloader, criterion, optimizer, scheduler, device, epoch, logger):\n    """Train for one epoch."""\n    model.train()\n    running_loss = 0.0\n    all_preds = []\n    all_labels = []\n\n    pbar = tqdm(dataloader, desc=f\'Epoch {epoch} [Train]\')\n\n    for images, labels in pbar:\n        images = images.to(device)\n        labels = labels.to(device)\n\n        # Forward pass\n        optimizer.zero_grad()\n        outputs = model(images)\n        loss = criterion(outputs, labels)\n\n        # Backward pass\n        loss.backward()\n        optimizer.step()\n        scheduler.step()\n\n        # Metrics\n        running_loss += loss.item() * images.size(0)\n        preds = torch.argmax(outputs, dim=1).cpu().numpy()\n        all_preds.extend(preds)\n        all_labels.extend(labels.cpu().numpy())\n\n        # Update progress bar\n        pbar.set_postfix({\'loss\': f\'{loss.item():.4f}\'})\n\n    epoch_loss = running_loss / len(dataloader.dataset)\n    epoch_acc = accuracy_score(all_labels, all_preds)\n\n    return epoch_loss, epoch_acc\n\n\ndef validate_epoch(model, dataloader, criterion, device, epoch, logger):\n    """Validate for one epoch."""\n    model.eval()\n    running_loss = 0.0\n    all_preds = []\n    all_labels = []\n\n    pbar = tqdm(dataloader, desc=f\'Epoch {epoch} [Val]\')\n\n    with torch.no_grad():\n        for images, labels in pbar:\n            images = images.to(device)\n            labels = labels.to(device)\n\n            outputs = model(images)\n            loss = criterion(outputs, labels)\n\n            running_loss += loss.item() * images.size(0)\n            preds = torch.argmax(outputs, dim=1).cpu().numpy()\n            all_preds.extend(preds)\n            all_labels.extend(labels.cpu().numpy())\n\n            pbar.set_postfix({\'loss\': f\'{loss.item():.4f}\'})\n\n    epoch_loss = running_loss / len(dataloader.dataset)\n    epoch_acc = accuracy_score(all_labels, all_preds)\n    conf_mat = confusion_matrix(all_labels, all_preds)\n\n    return epoch_loss, epoch_acc, conf_mat\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description=\'Train DeepFake Detection Model\',\n        formatter_class=argparse.ArgumentDefaultsHelpFormatter\n    )\n\n    parser.add_argument(\'--config\', type=str, default=None,\n                        help=\'Path to config file\')\n    parser.add_argument(\'--train-real\', type=str, nargs=\'+\', required=True,\n                        help=\'Paths to training real images directories\')\n    parser.add_argument(\'--train-fake\', type=str, nargs=\'+\', required=True,\n                        help=\'Paths to training fake images directories\')\n    parser.add_argument(\'--val-real\', type=str, nargs=\'+\', required=True,\n                        help=\'Paths to validation real images directories\')\n    parser.add_argument(\'--val-fake\', type=str, nargs=\'+\', required=True,\n                        help=\'Paths to validation fake images directories\')\n    parser.add_argument(\'--output-dir\', type=str, default=\'outputs\',\n                        help=\'Output directory for checkpoints and logs\')\n    parser.add_argument(\'--batch-size\', type=int, default=8,\n                        help=\'Batch size\')\n    parser.add_argument(\'--num-workers\', type=int, default=4,\n                        help=\'Number of DataLoader workers\')\n    parser.add_argument(\'--epochs\', type=int, default=20,\n                        help=\'Number of epochs\')\n    parser.add_argument(\'--lr\', type=float, default=8e-4,\n                        help=\'Learning rate\')\n    parser.add_argument(\'--model\', type=str, default=\'efficientnet-b1\',\n                        help=\'Model architecture\')\n    parser.add_argument(\'--multistream\', action=\'store_true\',\n                        help=\'Use tri-stream RGB+frequency+SRM-like detector\')\n    parser.add_argument(\'--freq-model\', type=str, default=\'efficientnet-b4\',\n                        help=\'[Deprecated — ignored] Frequency-stream backbone. \'\n                             \'New architecture uses --model for all three streams.\')\n    parser.add_argument(\'--srm-model\', type=str, default=\'efficientnet-b4\',\n                        help=\'[Deprecated — ignored] SRM-stream backbone. \'\n                             \'New architecture uses --model for all three streams.\')\n    parser.add_argument(\'--srm-filters\', type=int, default=30,\n                        help=\'Number of learnable SRM filters (noise channels)\')\n    parser.add_argument(\'--amp\', action=\'store_true\',\n                        help=\'Enable mixed precision (recommended on GPU)\')\n    parser.add_argument(\'--resume\', type=str, default=None,\n                        help=\'Resume from checkpoint\')\n    parser.add_argument(\'--balanced-sampler\', action=\'store_true\',\n                        help=\'Use WeightedRandomSampler to balance classes in each epoch (train only)\')\n    parser.add_argument(\'--focal-loss\', action=\'store_true\',\n                        help=\'Use Focal Loss instead of CrossEntropyLoss \'\n                             \'(recommended when combined with --balanced-sampler)\')\n    parser.add_argument(\'--focal-gamma\', type=float, default=2.0,\n                        help=\'Gamma exponent for Focal Loss (default 2.0; higher = more focus on hard examples)\')\n    parser.add_argument(\'--best-metric\', type=str, default=\'auc\',\n                        choices=[\'acc\', \'auc\', \'f1\'],\n                        help=\'Metric to select best_model (computed on val set)\')\n    parser.add_argument(\'--f1-threshold\', type=float, default=0.5,\n                        help=\'Threshold for F1 computation (on prob_real)\')\n\n    args = parser.parse_args()\n\n    # Create output directories\n    checkpoint_dir = Path(args.output_dir) / \'checkpoints\'\n    log_dir = Path(args.output_dir) / \'logs\'\n    results_dir = Path(args.output_dir) / \'results\'\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    log_dir.mkdir(parents=True, exist_ok=True)\n    results_dir.mkdir(parents=True, exist_ok=True)\n\n    # Setup logger\n    logger = setup_logger(\n        name=\'training\',\n        log_file=str(log_dir / \'training.log\'),\n        level=\'INFO\'\n    )\n\n    logger.info("="*60)\n    logger.info("DeepFake Detection Training")\n    logger.info("="*60)\n\n    # Device\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    logger.info(f"Using device: {device}")\n\n    # Create datasets\n    logger.info("Creating datasets...")\n\n    image_size = _effnet_input_size(args.model)\n\n    train_transforms = get_train_transforms(image_size)\n    val_transforms = get_val_transforms(image_size)\n\n    # Prepare data configs\n    train_real_config = [(path, -1) for path in args.train_real]\n    train_fake_config = [(path, -1) for path in args.train_fake]\n    val_real_config = [(path, -1) for path in args.val_real]\n    val_fake_config = [(path, -1) for path in args.val_fake]\n\n    train_dataset = create_combined_dataset(train_real_config, train_fake_config, train_transforms)\n    val_dataset = create_combined_dataset(val_real_config, val_fake_config, val_transforms)\n\n    logger.info(f"Train dataset: {len(train_dataset)} samples")\n    logger.info(f"Val dataset: {len(val_dataset)} samples")\n\n    # Create dataloaders\n    if args.multistream:\n        # Tri-stream is GPU-memory heavy; keep the default batch small.\n        args.batch_size = min(args.batch_size, 2)\n\n    train_sampler = None\n    if args.balanced_sampler:\n        # CombinedDataset exposes a DataFrame with a \'real\' column (1 for real, 0 for fake).\n        labels_np = np.asarray(train_dataset.data[\'real\'].values, dtype=np.int64)\n        class_counts = np.bincount(labels_np, minlength=2).astype(np.float64)\n        class_counts[class_counts == 0] = 1.0\n        class_weights = 1.0 / class_counts\n        sample_weights = class_weights[labels_np]\n        train_sampler = WeightedRandomSampler(\n            weights=torch.as_tensor(sample_weights, dtype=torch.double),\n            num_samples=len(sample_weights),\n            replacement=True,\n        )\n        logger.info(\n            f"Using balanced sampler: counts(fake={int(class_counts[0])}, real={int(class_counts[1])})"\n        )\n\n    train_loader, val_loader, _ = create_dataloaders(\n        train_dataset=train_dataset,\n        val_dataset=val_dataset,\n        batch_size=args.batch_size,\n        num_workers=args.num_workers,\n        train_sampler=train_sampler,\n    )\n\n    # Create model\n    if args.multistream:\n        logger.info(\n            f"Creating TriStreamDeepFakeDetector — SVG architecture\\n"\n            f"  Shared backbone : {args.model} (all 3 streams)\\n"\n            f"  Feature dim     : {_effnet_feat_dim(args.model)}-d\\n"\n            f"  Fusion          : Channel Attention (feature-level)\\n"\n            f"  Head            : LayerNorm → Dropout → FC(512) → GELU → FC(2)"\n        )\n        if args.freq_model != "efficientnet-b4" or args.srm_model != "efficientnet-b4":\n            logger.warning(\n                "--freq-model / --srm-model are deprecated in the new SVG architecture "\n                "and will be ignored. All three streams use --model as the shared backbone."\n            )\n        model = TriStreamDeepFakeDetector(\n            rgb_model=args.model,\n            srm_filters=args.srm_filters,\n            pretrained=True,\n        )\n    else:\n        logger.info(f"Creating DeepFakeDetector: {args.model}")\n        model = DeepFakeDetector(model_name=args.model, pretrained=True)\n    model = model.to(device)\n\n    total_params, trainable_params = model.count_parameters()\n    logger.info(f"Total parameters: {total_params:,}")\n    logger.info(f"Trainable parameters: {trainable_params:,}")\n\n    # Loss, optimizer, scheduler\n    if args.focal_loss:\n        criterion = FocalLoss(gamma=args.focal_gamma)\n        logger.info(\n            f"Loss function : FocalLoss (gamma={args.focal_gamma})"\n        )\n    else:\n        criterion = nn.CrossEntropyLoss()\n        logger.info("Loss function : CrossEntropyLoss")\n    optimizer = AdamW(model.parameters(), lr=args.lr, weight_decay=1e-3)\n\n    num_train_steps = len(train_loader) * args.epochs\n    scheduler = get_cosine_schedule_with_warmup(\n        optimizer,\n        num_warmup_steps=len(train_loader) * 5,\n        num_training_steps=num_train_steps\n    )\n\n    # Resume from checkpoint\n    start_epoch = 0\n    if args.resume:\n        logger.info(f"Resuming from checkpoint: {args.resume}")\n        ckpt = model.load_checkpoint(args.resume, device=str(device))\n        if isinstance(ckpt, dict):\n            if \'optimizer_state_dict\' in ckpt:\n                optimizer.load_state_dict(ckpt[\'optimizer_state_dict\'])\n                logger.info("Loaded optimizer state from checkpoint")\n            if \'scheduler_state_dict\' in ckpt:\n                scheduler.load_state_dict(ckpt[\'scheduler_state_dict\'])\n                logger.info("Loaded scheduler state from checkpoint")\n            if \'epoch\' in ckpt:\n                start_epoch = int(ckpt[\'epoch\'])\n                logger.info(f"Resuming from epoch {start_epoch + 1}")\n\n    # Training history\n    history = {\n        \'train_loss\': [],\n        \'train_accuracy\': [],\n        \'val_loss\': [],\n        \'val_accuracy\': []\n    }\n\n    # Training loop\n    best_val_acc = 0.0\n\n    scaler = torch.cuda.amp.GradScaler(enabled=bool(args.amp and device.type == "cuda"))\n\n    for epoch in range(start_epoch + 1, args.epochs + 1):\n        logger.info(f"\\nEpoch {epoch}/{args.epochs}")\n\n        # Train\n        # Train with optional AMP\n        model.train()\n        running_loss = 0.0\n        all_preds = []\n        all_labels = []\n        all_logits = []\n        pbar = tqdm(train_loader, desc=f\'Epoch {epoch} [Train]\')\n\n        for images, labels in pbar:\n            images = images.to(device, non_blocking=True)\n            labels = labels.to(device, non_blocking=True)\n\n            optimizer.zero_grad(set_to_none=True)\n\n            with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):\n                outputs = model(images)\n                loss = criterion(outputs, labels)\n\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n\n            running_loss += loss.item() * images.size(0)\n            preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()\n            all_preds.extend(preds)\n            all_labels.extend(labels.detach().cpu().numpy())\n\n            pbar.set_postfix({\'loss\': f\'{loss.item():.4f}\'})\n\n        train_loss = running_loss / len(train_loader.dataset)\n        train_acc = accuracy_score(all_labels, all_preds)\n\n        # Validate\n        model.eval()\n        running_loss = 0.0\n        all_preds = []\n        all_labels = []\n\n        pbar = tqdm(val_loader, desc=f\'Epoch {epoch} [Val]\')\n        with torch.no_grad():\n            for images, labels in pbar:\n                images = images.to(device, non_blocking=True)\n                labels = labels.to(device, non_blocking=True)\n\n                outputs = model(images)\n                loss = criterion(outputs, labels)\n\n                running_loss += loss.item() * images.size(0)\n                logits_np = outputs.detach().cpu().numpy()\n                all_logits.append(logits_np)\n                preds = np.argmax(logits_np, axis=1)\n                all_preds.extend(preds)\n                all_labels.extend(labels.detach().cpu().numpy())\n                pbar.set_postfix({\'loss\': f\'{loss.item():.4f}\'})\n\n        val_loss = running_loss / len(val_loader.dataset)\n        val_acc = accuracy_score(all_labels, all_preds)\n        conf_mat = confusion_matrix(all_labels, all_preds)\n\n        # Prob of REAL (class index 1) for AUC/F1/EER style metrics.\n        logits_all = np.concatenate(all_logits, axis=0) if all_logits else np.zeros((0, 2), dtype=np.float32)\n        probs = softmax(logits_all.astype(np.float64), axis=1)[:, 1] if logits_all.size else np.zeros((0,), dtype=np.float64)\n        labels_arr = np.asarray(all_labels, dtype=np.int64)\n        val_metrics = calculate_comprehensive_metrics(probs=probs, labels=labels_arr)\n        val_auc = float(val_metrics.get(\'auc_roc\', 0.0))\n        thr = float(args.f1_threshold)\n        val_f1 = float(f1_score(labels_arr, (probs >= thr).astype(np.int64))) if labels_arr.size else 0.0\n\n        # Metrics at optimal threshold (recommended for imbalanced datasets).\n        optimal_thr = float(val_metrics.get(\'optimal_threshold\', thr))\n        val_f1_opt = float(f1_score(labels_arr, (probs >= optimal_thr).astype(np.int64))) if labels_arr.size else 0.0\n        val_acc_opt = float(accuracy_score(labels_arr, (probs >= optimal_thr).astype(np.int64))) if labels_arr.size else 0.0\n\n        # Log metrics\n        logger.info(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")\n        logger.info(\n            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, "\n            f"Val AUC: {val_auc:.4f}, "\n            f"Val F1@{args.f1_threshold:.2f}: {val_f1:.4f}, "\n            f"Val F1@optThr({optimal_thr:.4f}): {val_f1_opt:.4f}, "\n            f"Val Acc@optThr({optimal_thr:.4f}): {val_acc_opt:.4f}"\n        )\n\n        # Update history\n        history[\'train_loss\'].append(train_loss)\n        history[\'train_accuracy\'].append(train_acc)\n        history[\'val_loss\'].append(val_loss)\n        history[\'val_accuracy\'].append(val_acc)\n\n        # Save checkpoint\n        checkpoint_path = checkpoint_dir / f\'epoch_{epoch}.pth\'\n        model.save_checkpoint(\n            str(checkpoint_path),\n            epoch=epoch,\n            optimizer_state=optimizer.state_dict(),\n            scheduler_state=scheduler.state_dict(),\n            metrics={\n                \'val_acc\': val_acc,\n                \'val_loss\': val_loss,\n                \'val_auc\': val_auc,\n                \'val_f1\': val_f1,\n                \'val_f1_opt\': val_f1_opt,\n                \'val_acc_opt\': val_acc_opt,\n                \'val_optimal_threshold\': optimal_thr,\n            }\n        )\n\n        # Save best model\n        if args.best_metric == \'acc\':\n            score = val_acc\n        elif args.best_metric == \'f1\':\n            score = val_f1\n        else:\n            score = val_auc\n\n        if score > best_val_acc:\n            best_val_acc = score\n            best_path = checkpoint_dir / \'best_model.pth\'\n            model.save_checkpoint(str(best_path), epoch=epoch)\n            logger.info(f"Best model saved with {args.best_metric}: {best_val_acc:.4f}")\n\n    # Plot training history\n    logger.info("Plotting training history...")\n    plot_training_history(\n        history,\n        metrics=[\'loss\', \'accuracy\'],\n        save_path=str(results_dir / \'training_history.png\'),\n        show=False\n    )\n\n    logger.info("Training complete!")\n    logger.info(f"Best validation {args.best_metric}: {best_val_acc:.4f}")\n\n\nif __name__ == \'__main__\':\n    main()\n')
print(f'  Ghi: /content/scripts/train.py')

os.makedirs('/content/scripts', exist_ok=True)
open('/content/scripts/test.py', 'w', encoding='utf-8').write('#!/usr/bin/env python3\n"""\nTesting and Evaluation Script for DeepFake Detection\nComprehensive evaluation with multiple metrics.\n"""\n\nimport argparse\nimport os\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).parent.parent))\n\nimport torch\nimport numpy as np\nfrom tqdm import tqdm\nfrom scipy.special import softmax\nimport pandas as pd\nimport os\n\nfrom deepfake_detector.models import DeepFakeDetector, TriStreamDeepFakeDetector\nfrom deepfake_detector.data import create_combined_dataset, get_val_transforms, create_dataloaders\nfrom deepfake_detector.utils import (\n    setup_logger, calculate_comprehensive_metrics, print_metrics,\n    plot_confusion_matrix, plot_roc_curve\n)\nfrom sklearn.metrics import confusion_matrix, accuracy_score\n\n\ndef _effnet_input_size(model_name: str) -> int:\n    name = model_name.lower()\n    if "b0" in name:\n        return 224\n    if "b1" in name:\n        return 240\n    if "b2" in name:\n        return 260\n    if "b3" in name:\n        return 300\n    if "b4" in name:\n        return 380\n    if "b5" in name:\n        return 456\n    if "b6" in name:\n        return 528\n    if "b7" in name:\n        return 600\n    return 224\n\n\ndef test_model(model, dataloader, device, logger):\n    """Test the model and collect predictions."""\n    model.eval()\n\n    all_probs = []\n    all_labels = []\n    all_image_paths = []\n\n    logger.info("Running inference on test set...")\n\n    with torch.no_grad():\n        for batch in tqdm(dataloader, desc=\'Testing\'):\n            if len(batch) == 3:\n                images, labels, image_paths = batch\n            else:\n                images, labels = batch\n                image_paths = None\n            images = images.to(device)\n\n            outputs = model(images)\n            probs = softmax(outputs.cpu().numpy(), axis=1)\n\n            all_probs.append(probs)\n            all_labels.extend(labels.numpy())\n            if image_paths is not None:\n                all_image_paths.extend(list(image_paths))\n\n    all_probs = np.vstack(all_probs)\n    all_labels = np.array(all_labels)\n\n    return all_probs, all_labels, all_image_paths\n\n\ndef extract_video_id(image_path: str) -> str:\n    """\n    Recover source video id from extracted face filename.\n    Expected naming from extract_faces.py:\n      <video_stem>-<timestamp>.jpg\n    """\n    stem = Path(image_path).stem\n    if \'-\' in stem:\n        return stem.rsplit(\'-\', 1)[0]\n    return stem\n\n\ndef compute_video_level_metrics(pred_df: pd.DataFrame):\n    grouped = pred_df.groupby(\'video_id\').agg(\n        true_label=(\'true_label\', \'max\'),\n        prob_real=(\'prob_real\', \'mean\'),\n        prob_fake=(\'prob_fake\', \'mean\'),\n    ).reset_index()\n    grouped[\'pred_label\'] = (grouped[\'prob_real\'] >= 0.5).astype(int)\n    video_acc = accuracy_score(grouped[\'true_label\'].values, grouped[\'pred_label\'].values)\n    return grouped, video_acc\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description=\'Test DeepFake Detection Model\',\n        formatter_class=argparse.ArgumentDefaultsHelpFormatter\n    )\n\n    parser.add_argument(\'--test-real\', type=str, nargs=\'+\', required=True,\n                        help=\'Paths to test real images directories\')\n    parser.add_argument(\'--test-fake\', type=str, nargs=\'+\', required=True,\n                        help=\'Paths to test fake images directories\')\n    parser.add_argument(\'--checkpoint\', type=str, required=True,\n                        help=\'Path to model checkpoint\')\n    parser.add_argument(\'--output-dir\', type=str, default=\'test_results\',\n                        help=\'Output directory for results\')\n    parser.add_argument(\'--batch-size\', type=int, default=100,\n                        help=\'Batch size\')\n    parser.add_argument(\'--num-workers\', type=int, default=4,\n                        help=\'Number of DataLoader workers\')\n    parser.add_argument(\'--model\', type=str, default=\'efficientnet-b1\',\n                        help=\'Model architecture\')\n    parser.add_argument(\'--multistream\', action=\'store_true\',\n                        help=\'Use tri-stream RGB+frequency+SRM-like detector\')\n    parser.add_argument(\'--freq-model\', type=str, default=\'efficientnet-b2\',\n                        help=\'EfficientNet variant for frequency stream\')\n    parser.add_argument(\'--srm-model\', type=str, default=\'efficientnet-b0\',\n                        help=\'EfficientNet variant for SRM-like stream\')\n    parser.add_argument(\'--srm-filters\', type=int, default=30,\n                        help=\'Number of learnable SRM filters (noise channels)\')\n    parser.add_argument(\'--save-predictions\', action=\'store_true\',\n                        help=\'Save predictions to CSV\')\n    parser.add_argument(\'--video-level\', action=\'store_true\',\n                        help=\'Compute video-level metrics from frame predictions\')\n\n    args = parser.parse_args()\n\n    # Create output directory\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    # Setup logger\n    logger = setup_logger(\n        name=\'testing\',\n        log_file=str(output_dir / \'test.log\'),\n        level=\'INFO\'\n    )\n\n    logger.info("="*60)\n    logger.info("DeepFake Detection Testing")\n    logger.info("="*60)\n\n    # Device\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    logger.info(f"Using device: {device}")\n\n    # Create test dataset\n    logger.info("Creating test dataset...")\n\n    image_size = _effnet_input_size(args.model)\n    test_transforms = get_val_transforms(image_size)\n\n    test_real_config = [(path, -1) for path in args.test_real]\n    test_fake_config = [(path, -1) for path in args.test_fake]\n\n    test_dataset = create_combined_dataset(\n        test_real_config,\n        test_fake_config,\n        test_transforms,\n        return_paths=True\n    )\n    logger.info(f"Test dataset: {len(test_dataset)} samples")\n\n    # Create dataloader\n    _, _, test_loader = create_dataloaders(\n        test_dataset=test_dataset,\n        batch_size=args.batch_size,\n        num_workers=args.num_workers\n    )\n\n    # Load model\n    logger.info(f"Loading model from: {args.checkpoint}")\n    if args.multistream:\n        model = TriStreamDeepFakeDetector(\n            rgb_model=args.model,\n            freq_model=args.freq_model,\n            srm_model=args.srm_model,\n            srm_filters=args.srm_filters,\n            pretrained=False,\n        )\n    else:\n        model = DeepFakeDetector(model_name=args.model, pretrained=False)\n    model.load_checkpoint(args.checkpoint, device=str(device))\n    model = model.to(device)\n    model.eval()\n\n    # Test model\n    probs, labels, image_paths = test_model(model, test_loader, device, logger)\n\n    # Get predictions for real class (index 0 is fake, index 1 is real)\n    # Assuming model outputs [fake_prob, real_prob]\n    pred_labels = np.argmax(probs, axis=1)\n    real_probs = probs[:, 1]  # Probability of being real\n\n    # Calculate metrics\n    logger.info("\\nCalculating metrics...")\n    metrics = calculate_comprehensive_metrics(real_probs, labels, pred_labels)\n\n    # Print metrics\n    print_metrics(metrics, title="Test Results")\n\n    # Save metrics\n    metrics_file = output_dir / \'metrics.txt\'\n    with open(metrics_file, \'w\') as f:\n        f.write("Test Results\\n")\n        f.write("="*60 + "\\n")\n        for key, value in metrics.items():\n            f.write(f"{key}: {value}\\n")\n    logger.info(f"Metrics saved to: {metrics_file}")\n\n    # Plot confusion matrix\n    conf_mat = confusion_matrix(labels, pred_labels)\n    plot_confusion_matrix(\n        conf_mat,\n        class_names=[\'Fake\', \'Real\'],\n        title=\'Test Set Confusion Matrix\',\n        save_path=str(output_dir / \'confusion_matrix.png\'),\n        show=False\n    )\n    logger.info("Confusion matrix saved")\n\n    # Plot ROC curve\n    from deepfake_detector.utils.metrics import get_EER_states\n    EER, optimal_thr, FRR_list, FAR_list = get_EER_states(real_probs, labels)\n\n    plot_roc_curve(\n        FRR_list, FAR_list, EER,\n        title=f\'ROC Curve (EER={EER:.4f})\',\n        save_path=str(output_dir / \'roc_curve.png\'),\n        show=False\n    )\n    logger.info("ROC curve saved")\n\n    # Save predictions\n    if args.save_predictions:\n        predictions_df = pd.DataFrame({\n            \'image_path\': image_paths if image_paths else [None] * len(labels),\n            \'true_label\': labels,\n            \'pred_label\': pred_labels,\n            \'prob_fake\': probs[:, 0],\n            \'prob_real\': probs[:, 1]\n        })\n        if image_paths:\n            predictions_df[\'video_id\'] = predictions_df[\'image_path\'].map(extract_video_id)\n        predictions_file = output_dir / \'predictions.csv\'\n        predictions_df.to_csv(predictions_file, index=False)\n        logger.info(f"Predictions saved to: {predictions_file}")\n\n        if args.video_level and image_paths:\n            video_df, video_acc = compute_video_level_metrics(predictions_df)\n            video_file = output_dir / \'video_predictions.csv\'\n            video_df.to_csv(video_file, index=False)\n            logger.info(f"Video-level predictions saved to: {video_file}")\n\n            video_metrics_file = output_dir / \'video_metrics.txt\'\n            with open(video_metrics_file, \'w\') as f:\n                f.write("Video-level Results\\n")\n                f.write("="*60 + "\\n")\n                f.write(f"num_videos: {len(video_df)}\\n")\n                f.write(f"video_accuracy: {video_acc:.6f}\\n")\n            logger.info(f"Video-level metrics saved to: {video_metrics_file}")\n\n    logger.info("\\nTesting complete!")\n    logger.info(f"Results saved to: {output_dir}")\n\n\nif __name__ == \'__main__\':\n    main()\n')
print(f'  Ghi: /content/scripts/test.py')

sys.path.insert(0, '/content')
sys.path.insert(0, '/content/scripts')
print('\nCode du an da san sang tai /content/')


  Ghi: /content/deepfake_detector/__init__.py
  Ghi: /content/deepfake_detector/models/__init__.py
  Ghi: /content/deepfake_detector/models/efficientnet.py
  Ghi: /content/deepfake_detector/models/multistream.py
  Ghi: /content/deepfake_detector/data/__init__.py
  Ghi: /content/deepfake_detector/data/dataset.py
  Ghi: /content/deepfake_detector/data/transforms.py
  Ghi: /content/deepfake_detector/data/loader.py
  Ghi: /content/deepfake_detector/utils/__init__.py
  Ghi: /content/deepfake_detector/utils/metrics.py
  Ghi: /content/deepfake_detector/utils/logger.py
  Ghi: /content/deepfake_detector/utils/visualization.py
  Ghi: /content/deepfake_detector/config/__init__.py
  Ghi: /content/deepfake_detector/config/config.py
  Ghi: /content/scripts/extract_faces.py
  Ghi: /content/scripts/extract_from_manifest.py
  Ghi: /content/scripts/create_split_manifest_from_csv.py
  Ghi: /content/scripts/train.py
  Ghi: /content/scripts/test.py

Code du an da san sang tai /content/


In [8]:
!pip install --upgrade --force-reinstall numpy
!pip install --upgrade --force-reinstall opencv-python-headless scipy pandas
# hoặc nếu cần cố định phiên bản numpy tương thích, ví dụ:
!pip install --upgrade --force-reinstall numpy==1.23.5

  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
facenet-pytorch 2.6.0 requires numpy<2.0.0,>=1.24.0, but you have numpy 2.4.3 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.3 MB/s eta 0:00:00
  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 61.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 121.9 MB/s eta 0:00:0000:010:01
Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 25.1 MB/s eta 0:00:00
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:
      Successfully uninstalled numpy-2.4.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 116.7 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [1]:
import sys
sys.path.insert(0, '/content')
from deepfake_detector.models import TriStreamDeepFakeDetector, DeepFakeDetector
from deepfake_detector.data import create_combined_dataset, get_train_transforms, get_val_transforms, create_dataloaders
from deepfake_detector.utils import setup_logger, calculate_comprehensive_metrics
print("Import OK — code du an hoat dong binh thuong")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

Import OK — code du an hoat dong binh thuong


---
## Phần 2 — Chuẩn bị dữ liệu

Tạo split manifest từ CSV rồi extract face crops từ video.  
**Kết quả lưu vào Drive của bạn** (`deepfake_work/`) — không mất khi Colab reset.

> **Bỏ qua phần này nếu đã extract xong** (chuyển thẳng xuống Phần 3).

In [2]:
# Tao split manifest 72/14/14 tu CSV
!python /content/scripts/create_split_manifest_from_csv.py \
    --dataset-root {DATASET_ROOT} \
    --csv-dir      {CSV_DIR} \
    --train-ratio  0.72 \
    --val-ratio    0.14 \
    --test-ratio   0.14 \
    --seed         42 \
    --output       {MANIFEST_PATH}

Traceback (most recent call last):
  File "/content/scripts/create_split_manifest_from_csv.py", line 204, in <module>
    main()
  File "/content/scripts/create_split_manifest_from_csv.py", line 103, in main
    raise FileNotFoundError(f"Dataset root not found: {dataset_root}")
FileNotFoundError: Dataset root not found: /content/{DATASET_ROOT}


In [ ]:
import json
with open(MANIFEST_PATH) as f:
    m = json.load(f)
print(f"{'Class':25s}  {'train':>6s}  {'val':>6s}  {'test':>6s}")
print("-" * 50)
all_cls = sorted(m["data"]["train"].keys())
for cls in all_cls:
    tr = len(m["data"]["train"].get(cls, []))
    vl = len(m["data"]["val"].get(cls, []))
    te = len(m["data"]["test"].get(cls, []))
    print(f"{cls:25s}  {tr:6d}  {vl:6d}  {te:6d}")

In [ ]:
# Extract train faces (luu vao Drive)
!python /content/scripts/extract_from_manifest.py \
    --manifest               {MANIFEST_PATH} \
    --split                  train \
    --output-root            {EXTRACTED_ROOT} \
    --real-class             original \
    --real-frames-per-video  {FRAMES_PER_VIDEO} \
    --fake-frames-per-video  {FRAMES_PER_VIDEO} \
    --sampling-strategy      uniform \
    --device                 cuda

In [ ]:
# Extract val faces (luu vao Drive)
!python /content/scripts/extract_from_manifest.py \
    --manifest               {MANIFEST_PATH} \
    --split                  val \
    --output-root            {EXTRACTED_ROOT} \
    --real-class             original \
    --real-frames-per-video  {FRAMES_PER_VIDEO} \
    --fake-frames-per-video  {FRAMES_PER_VIDEO} \
    --sampling-strategy      uniform \
    --device                 cuda

In [ ]:
# Extract test faces (luu vao Drive)
!python /content/scripts/extract_from_manifest.py \
    --manifest               {MANIFEST_PATH} \
    --split                  test \
    --output-root            {EXTRACTED_ROOT} \
    --real-class             original \
    --real-frames-per-video  {FRAMES_PER_VIDEO} \
    --fake-frames-per-video  {FRAMES_PER_VIDEO} \
    --sampling-strategy      uniform \
    --device                 cuda

In [ ]:
import os
print(f"{'Split':6s}  {'Class':25s}  {'Anh':>7s}")
print("-" * 45)
for split in ["train", "val", "test"]:
    split_dir = os.path.join(EXTRACTED_ROOT, split)
    if not os.path.exists(split_dir):
        print(f"{split}: chua extract")
        continue
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        n = len([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg",".png",".jpeg"))])
        print(f"{split:6s}  {cls:25s}  {n:7,}")

---
## Phần 3 — Training

Sử dụng extracted face crops để train Tri-Stream model (~2-3 giờ trên T4).

In [ ]:
import os

TRAIN_REAL_DIR = os.path.join(EXTRACTED_ROOT, "train", REAL_CLASS)
VAL_REAL_DIR   = os.path.join(EXTRACTED_ROOT, "val",   REAL_CLASS)

TRAIN_FAKE_DIRS = " ".join([
    os.path.join(EXTRACTED_ROOT, "train", cls)
    for cls in FAKE_CLASSES
    if os.path.exists(os.path.join(EXTRACTED_ROOT, "train", cls))
])
VAL_FAKE_DIRS = " ".join([
    os.path.join(EXTRACTED_ROOT, "val", cls)
    for cls in FAKE_CLASSES
    if os.path.exists(os.path.join(EXTRACTED_ROOT, "val", cls))
])

EPOCHS      = 15
BATCH_SIZE  = 2     # T4 16GB: thu 4; A100: thu 8
LR          = "3e-4"
NUM_WORKERS = 2

print("Train REAL :", TRAIN_REAL_DIR)
print("Train FAKE :", TRAIN_FAKE_DIRS[:120], "...")
print("Output dir :", OUTPUT_DIR)

In [ ]:
import sys
sys.path.insert(0, '/content')

!python /content/scripts/train.py \
    --multistream \
    --model        efficientnet-b4 \
    --train-real   {TRAIN_REAL_DIR} \
    --train-fake   {TRAIN_FAKE_DIRS} \
    --val-real     {VAL_REAL_DIR} \
    --val-fake     {VAL_FAKE_DIRS} \
    --output-dir   {OUTPUT_DIR} \
    --epochs       {EPOCHS} \
    --batch-size   {BATCH_SIZE} \
    --lr           {LR} \
    --num-workers  {NUM_WORKERS} \
    --balanced-sampler \
    --focal-loss \
    --focal-gamma  2.0 \
    --amp

In [ ]:
import os
log_file = os.path.join(OUTPUT_DIR, "logs", "training.log")
if os.path.exists(log_file):
    with open(log_file) as f:
        lines = f.readlines()
    for l in lines[-30:]:
        print(l, end="")
else:
    print("Log chua co, chay training truoc")

---
## Phần 4 — Evaluation trên tập Test

In [ ]:
import os, glob

BEST_MODEL = os.path.join(OUTPUT_DIR, "checkpoints", "best_model.pth")
if not os.path.exists(BEST_MODEL):
    ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoints", "epoch_*.pth")))
    BEST_MODEL = ckpts[-1] if ckpts else None

if BEST_MODEL:
    print(f"Su dung checkpoint: {BEST_MODEL}")
else:
    print("Chua co checkpoint, hay chay training truoc")

In [ ]:
TEST_REAL_DIR  = os.path.join(EXTRACTED_ROOT, "test", REAL_CLASS)
TEST_FAKE_DIRS = " ".join([
    os.path.join(EXTRACTED_ROOT, "test", cls)
    for cls in FAKE_CLASSES
    if os.path.exists(os.path.join(EXTRACTED_ROOT, "test", cls))
])
TEST_OUT = os.path.join(OUTPUT_DIR, "test_results")
os.makedirs(TEST_OUT, exist_ok=True)
print("Test REAL :", TEST_REAL_DIR)
print("Test FAKE :", TEST_FAKE_DIRS[:120], "...")

In [ ]:
import sys
sys.path.insert(0, '/content')

!python /content/scripts/test.py \
    --multistream \
    --model       efficientnet-b4 \
    --test-real   {TEST_REAL_DIR} \
    --test-fake   {TEST_FAKE_DIRS} \
    --checkpoint  {BEST_MODEL} \
    --output-dir  {TEST_OUT} \
    --batch-size  32 \
    --num-workers {NUM_WORKERS} \
    --save-predictions \
    --video-level

In [ ]:
import os
metrics_file = os.path.join(TEST_OUT, "metrics.txt")
if os.path.exists(metrics_file):
    with open(metrics_file) as f:
        print(f.read())
else:
    print("Chua co metrics, chay evaluation truoc")

In [ ]:
from IPython.display import Image as IPImage, display
import os

for name in ["confusion_matrix.png", "roc_curve.png"]:
    p = os.path.join(TEST_OUT, name)
    if os.path.exists(p):
        print(f"=== {name} ===")
        display(IPImage(p))
    else:
        print(f"{name} chua co")

In [12]:
# Cell: Chẩn đoán và sửa lỗi NumPy không tương thích
# - In ra phiên bản numpy hiện tại
# - Ép cài lại numpy và các gói biên dịch thường gặp
# - Khởi động lại runtime để thay đổi có hiệu lực
import sys, os, subprocess

print('=== Phiên bản NumPy hiện tại ===')
try:
    import numpy as np
    print(np.__version__)
except Exception as e:
    print('Không thể import numpy:', e)

print('\n=== Bắt đầu cài lại NumPy và các gói liên quan ===')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall', 'numpy'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall', 'opencv-python-headless', 'scipy', 'pandas'])

print('\nHoàn tất cài đặt. Notebook sẽ khởi động lại runtime để áp dụng thay đổi...')
# Kill the process to force Colab runtime restart
os.kill(os.getpid(), 9)


=== Phiên bản NumPy hiện tại ===
2.0.2

=== Bắt đầu cài lại NumPy và các gói liên quan ===


: 

: 

: 

In [1]:
# Cell: Chẩn đoán và sửa lỗi NumPy không tương thích
# - In ra phiên bản numpy hiện tại
# - Ép cài lại numpy và các gói biên dịch thường gặp
# - Khởi động lại runtime để thay đổi có hiệu lực
import sys, os, subprocess

print('=== Phiên bản NumPy hiện tại ===')
try:
    import numpy as np
    print(np.__version__)
except Exception as e:
    print('Không thể import numpy:', e)

print('\n=== Bắt đầu cài lại NumPy và các gói liên quan ===')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall', 'numpy'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall', 'opencv-python-headless', 'scipy', 'pandas'])

print('\nHoàn tất cài đặt. Notebook sẽ khởi động lại runtime để áp dụng thay đổi...')
# Kill the process to force Colab runtime restart
os.kill(os.getpid(), 9)


=== Phiên bản NumPy hiện tại ===
2.4.3

=== Bắt đầu cài lại NumPy và các gói liên quan ===


: 

: 

: 